<a href="https://colab.research.google.com/github/hongfeiww/DALAS/blob/main/DALAS_TP_MegaStore_students_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MegaStore Lab -STUDENTS

## Context

This lab considers MegaStore, a retail chain that sells groceries and home goods across six physical stores in France and an online shop. Its computer systems were added one by one over time without any connection between them. A point-of-sale system runs the checkouts. A separate platform runs the website. Customers are kept in a dedicated CRM. A ticketing system logs customer service issues. The stores send daily inventory reports.

The marketing team needs to identify customers likely to stop buying in time to take action. Answering this question requires bringing five separate systems into a single, coherent view of the customer. The final dataset must be clean and reliable enough to train a predictive model.

## Project Scenario

This lab builds a single customer view from raw, unverified source data. Before training a model, the workflow addresses the necessary preparation steps: inspecting table contents, resolving missing or conflicting records, assembling a fact table, and creating a dataset restricted to what was known at the prediction date. This dataset is divided into training and test sets, alongside a dedicated scoring table.


## Objectives
Construct two tables from raw data:

1. **`mart.training_churn` (Training/Test Dataset):** A table with a granularity of one row per `(crm_ref, obs_date)`. Features are computed from transaction records preceding the observation date (`sale_date < obs_date - 2 days`). The target variable is computed from transaction records on or after the observation date (`sale_date >= obs_date`).
2. **`mart.scoring_churn` (Scoring Dataset):** The same feature columns as `mart.training_churn`, computed for a later date.


## Lab Organisation

Each part of the lab uses the output of the previous one.

**Part 1: Raw Data Audit**  
 Examines the source files before any transformation: determining what a single sales row represents, what `NULL` values indicate, and how many transactions link to missing stores or customers.

**Part 2: Dimension Tables**  
Constructs the store, product, and customer dimensions. The customer dimension tracks historical profile changes over time.

**Part 3: Fact Table Assembly**  
Combines the sales data with the dimensions to create a central fact table at the line-item level (one row per product sold). It resolves all foreign keys, including unmatched stores and customers, into valid dimension references. All later steps query this table.

**Part 4: Training Dataset Construction**  
Uses the fact table to build the model training set across multiple observation dates. Features are computed strictly from historical purchases, while labels are derived from subsequent activity. This step strictly separates past events from future outcomes to prevent data leakage.

**Part 5: Scoring Dataset and Leakage Impact**  
Applies the identical feature logic to a new date after the training window, generating a table used for inference. It also quantifies what happens when temporal boundaries are ignored, measuring the distortion caused by data leakage.




# Setup (Provided)

Schema layout and function glossary.

**The three database schemas** (course: *Database Organization: Schemas and the Load Process*):

| Schema | Content | Written by |
|---|---|---|
| `raw` | views over the source files, never modified | ingestion |
| `stg` | intermediate cleaning steps | this lab |
| `mart` | dimensions, fact table, training and scoring tables | this lab |

The full data dictionary is in  **`DATA_DICTIONARY.md`**.

### Packages

In [1]:
!pip -q install duckdb pandas scikit-learn matplotlib

### Cell 0.2: Synthetic data sources (Provided)

Generates the eight files in `megastore_data/`. See their description in **`DATA_DICTIONARY.md`**.

In [2]:
# Cell 0.2
import csv, json, os, random, datetime as dt

random.seed(7)
OUT = "megastore_data"
os.makedirs(OUT, exist_ok=True)
START = dt.date(2024, 1, 1)
END = dt.date(2025, 12, 31)

stores = [
    ("M01", "MegaStore Strasbourg", "Strasbourg", "Grand Est"),
    ("M02", "MegaStore Paris Bastille", "Paris", "Ile-de-France"),
    ("M03", "MegaStore Lyon Part-Dieu", "Lyon", "Auvergne-Rhone-Alpes"),
    ("M04", "MegaStore Toulouse", "Toulouse", "Occitanie"),
    ("M05", "MegaStore Lille", "Lille", "Hauts-de-France"),
    ("M06", "MegaStore Nantes", "Nantes", "Pays de la Loire"),
]
with open(f"{OUT}/stores.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["store_code","store_name","city","region"]); w.writerows(stores)

# Product catalog
cats = {
 "Beverages": [("Still Water 1L","Cristalline",0.40,1.10),("Orange Juice 1L","Tropicana",0.90,2.30),
               ("Cola Soda 1.5L","CocaCola",0.70,1.80),("Ground Coffee 250g","Carte Noire",1.50,3.40),
               ("Green Tea 20 bags","Lipton",0.80,2.10)],
 "Grocery": [("Pasta 500g","Barilla",0.50,1.20),("Rice 1kg","Uncle Bens",1.10,2.50),
             ("Tomato Sauce 400g","Panzani",0.60,1.40),("Flour 1kg","Francine",0.45,1.15),
             ("Sugar 1kg","Daddy",0.65,1.50),("Olive Oil 75cl","Puget",3.20,7.80)],
 "Hygiene": [("Toothpaste 75ml","Signal",0.80,2.20),("Liquid Soap 300ml","Dove",1.10,2.90),
             ("Shampoo 250ml","Elseve",1.30,3.50),("Toilet Paper x12","Lotus",2.00,4.80)],
 "Fresh": [("Milk 1L","Lactel",0.55,1.25),("Butter 250g","President",1.20,2.60),
           ("Plain Yogurt x8","Danone",1.10,2.40),("Grated Cheese 200g","Entremont",1.00,2.30)],
 "Snacks": [("Chips 150g","Lays",0.60,1.70),("Biscuits 300g","LU",0.90,2.20),
            ("Dark Chocolate 100g","Lindt",1.10,2.80),("Cereal 500g","Kelloggs",1.40,3.60)],
}
products, ean = [], 3000000000000
for cat, items in cats.items():
    for lib, brand, cost, price in items:
        ean += random.randint(11, 97)
        style = random.choice([str.upper, str.lower, lambda s: s, lambda s: "  " + s + " "])
        products.append((str(ean), style(lib), brand, cat, cost, price))
with open(f"{OUT}/products.csv","w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["barcode","product_name","brand","category","unit_cost","unit_price"]); w.writerows(products)

barcodes=[p[0] for p in products]; price_map={p[0]:p[5] for p in products}

# CRM (Customer Relationship Management) demographic snapshots: crm_customers.csv
first=["Jean","Marie","Paul","Sophie","Luc","Emma","Hugo","Lea","Yanis","Chloe","Karim","Julie","Thomas","Nadia","Antoine","Fatima","Louis","Camille"]
last=["Durand","Martin","Bernard","Petit","Robert","Moreau","Girard","Lefevre","Faure","Roux","Fontaine","Chevalier","Da Silva","Nguyen","Benali"]
cities=[("Strasbourg","67"),("Paris","75"),("Lyon","69"),("Toulouse","31"),("Lille","59"),("Nantes","44"),("Metz","57"),("Bordeaux","33")]
segments=["Standard","Pro","VIP"]
customers=[]
for i in range(1,301):
    ref=f"CLI-{4000+i}"; fn=random.choice(first); ln=random.choice(last); c,d=random.choice(cities)
    forms=[f"{ln.upper()} {fn[0].lower()}.", f"{fn} {ln}", f"{fn.lower()} {ln.upper()}", f"  {ln} {fn}  ", f"{ln}, {fn}"]
    g=random.choice(["M","F","1","2","m","f"])
    city=f"{c} ({d})" if random.random()<0.6 else c
    customers.append([ref, random.choice(forms), g, city, random.choice(segments)])
with open(f"{OUT}/crm_customers.csv","w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["crm_ref","full_name","gender","city","segment"]); w.writerows(customers)

#crm_customers_d2.csv
changed=sorted(random.sample(range(len(customers)),40))
cust_d2=[r[:] for r in customers]
for idx in changed:
    c,d=random.choice(cities); cust_d2[idx][3]=f"{c} ({d})"; cust_d2[idx][4]="VIP"
with open(f"{OUT}/crm_customers_d2.csv","w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["crm_ref","full_name","gender","city","segment"]); w.writerows(cust_d2)

#pos_transactions.csv and web_orders.json
refs=[c[0] for c in customers]
CHURN_MIN=dt.date(2024,4,1); CHURN_MAX=dt.date(2025,11,1)
profiles={}
for ref in refs:
    lam=random.uniform(4.0,30.0)
    if random.random()<0.45:
        cd=CHURN_MIN+dt.timedelta(days=random.randint(0,(CHURN_MAX-CHURN_MIN).days))
    else:
        cd=None
    profiles[ref]=(lam,cd)

receipts=[]
for ref,(lam,cd) in profiles.items():
    d=START+dt.timedelta(days=random.randint(0,20))
    while True:
        gap=max(1,int(random.expovariate(lam/365.0)))
        d=d+dt.timedelta(days=gap)
        if d>END: break
        if cd is not None and d>cd: break
        receipts.append((ref,d))
receipts.sort(key=lambda x:x[1])


pos_rows=[]; ecom=[]; rid=100000; oid=900000
store_codes=[s[0] for s in stores]+["M99"]
devices=["mobile_ios","mobile_android","desktop_chrome","desktop_safari"]
payments=["credit_card","paypal","apple_pay"]
for ref,d in receipts:
    n_lines=random.choices([1,2,3,4,5],weights=[38,27,18,10,7])[0]
    web=random.random()<0.15
    picked=random.sample(barcodes,min(n_lines,len(barcodes)))
    if web:
        oid+=1
        for ln,cb in enumerate(picked,start=1):
            qv=random.choices([1,2,3],weights=[70,22,8])[0]
            ecom.append({"order_id":str(oid),"line_no":ln,"order_date":d.isoformat(),"sku":cb,
                         "customer_ref":ref,"qty":qv,"amount_eur":round(qv*price_map[cb],2),
                         "channel":"web","status":"cancelled" if random.random()<0.05 else None,
                         "metadata":{"device":random.choice(devices),"payment_method":random.choice(payments)}})
    else:
        rid+=1
        sc=random.choices(store_codes,weights=[16,16,16,16,16,16,4])[0]
        st="cancelled" if random.random()<0.07 else None
        for ln,cb in enumerate(picked,start=1):
            cb2="9999999999999" if random.random()<0.02 else cb
            qv=random.choices([1,2,3,4,5],weights=[50,25,12,8,5])[0]
            up=price_map.get(cb2,5.00)
            pos_rows.append([str(rid),ln,d.isoformat(),cb2,sc,ref,qv,int(round(qv*up*100)),st])

n=len(pos_rows)
for r in random.sample(pos_rows,int(0.02*n)): r[7]=""
for r in random.sample(pos_rows,int(0.02*n)): r[6]=random.choice([0,-1,-2])
for r in random.sample(pos_rows,int(0.006*n)): r[7]=random.randint(150000,900000)
for r in random.sample(pos_rows,int(0.03*n)): r[5]=""
dups=[r[:] for r in random.sample(pos_rows,int(0.012*n))]
allrows=pos_rows+dups
random.shuffle(allrows)
with open(f"{OUT}/pos_transactions.csv","w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["receipt_id","line_no","sale_date","barcode","store_code","crm_ref","quantity","amount_cents","status"]); w.writerows(allrows)
with open(f"{OUT}/web_orders.json","w",encoding="utf-8") as f:
    json.dump(ecom,f,ensure_ascii=False,indent=1)

#support_tickets.csv
heavy=random.sample(refs,8)
tickets=[]; tck=50000
for _ in range(700):
    tck+=1
    d=START+dt.timedelta(days=random.randint(0,(END-START).days))
    tickets.append([str(tck),d.isoformat(),random.choice(refs),random.choice(["Delivery Delay","Damaged Box","Wrong Item","Billing"]),random.choice([0,1])])
for ref in heavy:
    for _ in range(5):
        tck+=1
        d=START+dt.timedelta(days=random.randint(0,(END-START).days))
        tickets.append([str(tck),d.isoformat(),ref,"Billing",random.choice([0,1])])
random.shuffle(tickets)
with open(f"{OUT}/support_tickets.csv","w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["ticket_id","ticket_date","crm_ref","issue_type","is_resolved"]); w.writerows(tickets)

inv=[]
for k in range(60):
    cd=START+dt.timedelta(days=k)
    for sc in ["M01","M02","M03"]:
        for cb in barcodes[:4]:
            inv.append([cd.isoformat(),sc,cb,random.randint(20,120)])
with open(f"{OUT}/daily_inventory.csv","w",newline="",encoding="utf-8") as f:
    w=csv.writer(f); w.writerow(["snapshot_date","store_code","barcode","stock_units"]); w.writerows(inv)
# --- generator self-checks
import collections
_lines = collections.Counter(r[0] for r in pos_rows)
assert max(_lines.values()) >= 4, "receipts must carry several lines"
assert len(allrows) > len(pos_rows), "duplicate lines must be present"
assert any(r[4] == "M99" for r in pos_rows), "unregistered store code missing"
assert sum(1 for r in pos_rows if r[5] == "") > 0, "rows without customer reference missing"
_tk = collections.Counter(t[2] for t in tickets)
assert sum(1 for v in _tk.values() if v >= 4) >= 6, "not enough heavy ticket customers"
n_churned = sum(1 for _, cd in profiles.values() if cd is not None)
assert 0.30 < n_churned / len(profiles) < 0.60, "churn rate outside expected range"
print("Files written in", OUT)
print("POS lines:", len(allrows), "| web lines:", len(ecom), "| receipts:", len(receipts))

Files written in megastore_data
POS lines: 14828 | web lines: 2781 | receipts: 7843


### Cell 0.3 — Create the schemas and the raw views (Provided)

 Barcodes and receipt identifiers are stored as VARCHAR (allows to insert `Unknown` members in Part 2).

In [3]:
import duckdb, pandas as pd, numpy as np, hashlib, json, os, datetime as dt

con = duckdb.connect(":memory:")
def q(sql):
    return con.execute(sql).df()


def show_database_schema():
    query = """
    SELECT DISTINCT table_name
    FROM information_schema.columns
    WHERE table_schema='raw'
    ORDER BY table_name
    """
    return con.execute(query).df()

#Schema of one table
def show_table_schema(table_name):
    df_schema = con.execute(f"DESCRIBE {table_name}").df()

    columns_to_drop = [col for col in ['extra', 'default'] if col in df_schema.columns]
    if columns_to_drop:
        df_schema = df_schema.drop(columns=columns_to_drop)
    return df_schema


#Schemas of all tables
def show_table_schema_all():
    raw_tables = show_database_schema()
    for index, row in raw_tables.iterrows():
        table_name = f"raw.{row['table_name']}"
        print(f"\n--- Schema for {table_name} ---")
        display(show_table_schema(table_name))


#Row numbers of all tables
def show_row_numbers_all_tables():
    raw_tables = show_database_schema()
    for index, row in raw_tables.iterrows():
        table_name = f"raw.{row['table_name']}"
        print(f"\n--- Number of rows for {table_name} ---")
        display(con.execute(f"Select COUNT(*) FROM {table_name}").fetchone()[0])


for s in ["raw", "stg", "mart"]:
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {s}")

#Macro used to normalise text columns coming from the source files
#removes whitespace at the start and end, replacing multiple spaces within the text with a single space
con.execute("CREATE OR REPLACE MACRO clean_text(s) AS trim(regexp_replace(s, '\\s+', ' ', 'g'))")

con.execute("""CREATE OR REPLACE VIEW raw.pos_transactions AS SELECT * FROM read_csv('megastore_data/pos_transactions.csv',
 header=true, types={'receipt_id':'VARCHAR','line_no':'INTEGER','sale_date':'DATE','barcode':'VARCHAR',
 'store_code':'VARCHAR','crm_ref':'VARCHAR','quantity':'INTEGER','amount_cents':'INTEGER','status':'VARCHAR'})""")
con.execute("""CREATE OR REPLACE VIEW raw.web_orders AS SELECT * FROM read_json('megastore_data/web_orders.json', auto_detect=false, columns={
    'order_id': 'VARCHAR','line_no': 'INTEGER','order_date': 'DATE','sku': 'VARCHAR','customer_ref': 'VARCHAR','qty': 'INTEGER','amount_eur': 'DOUBLE',
    'channel': 'VARCHAR','status': 'VARCHAR','metadata': 'STRUCT(device VARCHAR, payment_method VARCHAR)'})""")
con.execute("""CREATE OR REPLACE VIEW raw.stores AS SELECT * FROM read_csv('megastore_data/stores.csv',
 header=true, types={'store_code':'VARCHAR','store_name':'VARCHAR','city':'VARCHAR','region':'VARCHAR'})""")
con.execute("""CREATE OR REPLACE VIEW raw.products AS SELECT * FROM read_csv('megastore_data/products.csv',
 header=true, types={'barcode':'VARCHAR','product_name':'VARCHAR','brand':'VARCHAR','category':'VARCHAR',
 'unit_cost':'DOUBLE','unit_price':'DOUBLE'})""")
con.execute("""CREATE OR REPLACE VIEW raw.crm_customers AS SELECT * FROM read_csv('megastore_data/crm_customers.csv',
 header=true, types={'crm_ref':'VARCHAR','full_name':'VARCHAR','gender':'VARCHAR','city':'VARCHAR','segment':'VARCHAR'})""")
con.execute("""CREATE OR REPLACE VIEW raw.crm_customers_d2 AS SELECT * FROM read_csv('megastore_data/crm_customers_d2.csv',
 header=true, types={'crm_ref':'VARCHAR','full_name':'VARCHAR','gender':'VARCHAR','city':'VARCHAR','segment':'VARCHAR'})""")
con.execute("""CREATE OR REPLACE VIEW raw.support_tickets AS SELECT * FROM read_csv('megastore_data/support_tickets.csv',
 header=true, types={'ticket_id':'VARCHAR','ticket_date':'DATE','crm_ref':'VARCHAR','issue_type':'VARCHAR','is_resolved':'INTEGER'})""")
con.execute("""CREATE OR REPLACE VIEW raw.daily_inventory AS SELECT * FROM read_csv('megastore_data/daily_inventory.csv',
 header=true, types={'snapshot_date':'DATE','store_code':'VARCHAR','barcode':'VARCHAR','stock_units':'INTEGER'})""")

In [4]:
show_database_schema()

,table_name
0,crm_customers
1,crm_customers_d2
2,daily_inventory
3,pos_transactions
4,products
5,stores
6,support_tickets
7,web_orders


In [5]:
show_table_schema_all()



--- Schema for raw.crm_customers ---


,column_name,column_type,null,key
0,crm_ref,VARCHAR,YES,None
1,full_name,VARCHAR,YES,None
2,gender,VARCHAR,YES,None
3,city,VARCHAR,YES,None
4,segment,VARCHAR,YES,None



--- Schema for raw.crm_customers_d2 ---


,column_name,column_type,null,key
0,crm_ref,VARCHAR,YES,None
1,full_name,VARCHAR,YES,None
2,gender,VARCHAR,YES,None
3,city,VARCHAR,YES,None
4,segment,VARCHAR,YES,None



--- Schema for raw.daily_inventory ---


...

,column_name,column_type,null,key
0,store_code,VARCHAR,YES,None
1,store_name,VARCHAR,YES,None
2,city,VARCHAR,YES,None
3,region,VARCHAR,YES,None



--- Schema for raw.support_tickets ---


,column_name,column_type,null,key
0,ticket_id,VARCHAR,YES,None
1,ticket_date,DATE,YES,None
2,crm_ref,VARCHAR,YES,None
3,issue_type,VARCHAR,YES,None
4,is_resolved,INTEGER,YES,None



--- Schema for raw.web_orders ---


,column_name,column_type,null,key
0,order_id,VARCHAR,YES,None
1,line_no,INTEGER,YES,None
2,order_date,DATE,YES,None
3,sku,VARCHAR,YES,None
4,customer_ref,VARCHAR,YES,None
5,qty,INTEGER,YES,None
6,amount_eur,DOUBLE,YES,None
7,channel,VARCHAR,YES,None
8,status,VARCHAR,YES,None
9,metadata,"STRUCT(device VARCHAR, payment_method VARCHAR)",YES,None


In [6]:
show_row_numbers_all_tables()


--- Number of rows for raw.crm_customers ---


300


--- Number of rows for raw.crm_customers_d2 ---


300


--- Number of rows for raw.daily_inventory ---


...

6


--- Number of rows for raw.support_tickets ---


740


--- Number of rows for raw.web_orders ---


2781

### Cell 0.4 — Functions used in this lab (Provided)

| Function | Usage in this lab | Documentation |
|---|---|---|
| `IS DISTINCT FROM` | comparison that treats NULL as a value | https://duckdb.org/docs/stable/sql/expressions/comparison_operators |
| `count(*) FILTER (WHERE ...)` | conditional count in one pass | https://duckdb.org/docs/stable/sql/query_syntax/filter |
| `arg_max(v, k)` | value of `v` at the row with the largest `k` | https://duckdb.org/docs/stable/sql/functions/aggregates |
| `OVER (PARTITION BY ... ORDER BY ...)` | Window clause: runs a calculation across each group defined by `PARTITION BY`, sorted by `ORDER BY`. Unlike `GROUP BY`, it keeps each individual row intact instead of collapsing them| https://duckdb.org/docs/stable/sql/functions/window_functions |
| `lag(x) OVER (PARTITION BY ... ORDER BY ...)` | previous row inside a group | https://duckdb.org/docs/stable/sql/functions/window_functions |
| `lead(x) OVER (...)` | next row inside a group | https://duckdb.org/docs/stable/sql/functions/window_functions |
| `row_number() OVER (...)` | position inside a group, used for de-duplication | https://duckdb.org/docs/stable/sql/functions/window_functions |
| `date_diff('day', a, b)` | signed difference in days | https://duckdb.org/docs/stable/sql/functions/date |
| `generate_series` / `unnest` | build a list of dates and expand it into rows | https://duckdb.org/docs/stable/sql/functions/list |
| `->>` | extract a text field from a JSON value | https://duckdb.org/docs/stable/data/json/json_functions |
| `SELECT * EXCLUDE (col)` | project every column but one | https://duckdb.org/docs/stable/sql/query_syntax/select |
| `split_part(s, sep, n)` | n-th fragment of a string | https://duckdb.org/docs/stable/sql/functions/text |
| `coalesce(a, b, ...)` | returns the first non-NULL argument. Used here to display NULL as a visible label rather than a blank cell | https://duckdb.org/docs/stable/sql/data_types/nulls |

---

# Part 1 — Properties of a source table

**Step 1:**.
**Course slides:** Section 1 - *Table Grain*, *Data Profiling* and *SQL Three-Valued Logic*, Section 2 -  *Silent Loss 1 and 2*, Section 3 - *Form 3, detecting rows dropped by a join*.

**Input:** `raw.pos_transactions`, `raw.stores`, `raw.crm_customers`.

**Output:** three diagnostics used in Part 3

Describe three properties for a source table: what one row represents (its grain), what NULL means in each column, and which foreign keys have no match. These properties influence the rows that are kept later in the pipeline.

### Task 1.1 — Grain of `raw.pos_transactions`

**Course slides:** Section 1- *Table Grain* and *Data Profiling: Verifying Table Grain Empirically*.

**TO DO:** Write one query returning four columns:

| Column | Content |
|---|---|
| `n_rows` | total number of rows |
| `n_receipts` | number of distinct `receipt_id` |
| `n_lines` | number of distinct pairs `(receipt_id, line_no)` |
| `max_lines_per_receipt` | largest number of rows sharing one `receipt_id` |

**Validation:** the three counts sould be different, and the maximum number of lines per receipt should be greater than 1.

In [7]:
show_table_schema('raw.pos_transactions')

,column_name,column_type,null,key
0,receipt_id,VARCHAR,YES,None
1,line_no,INTEGER,YES,None
2,sale_date,DATE,YES,None
3,barcode,VARCHAR,YES,None
4,store_code,VARCHAR,YES,None
5,crm_ref,VARCHAR,YES,None
6,quantity,INTEGER,YES,None
7,amount_cents,INTEGER,YES,None
8,status,VARCHAR,YES,None


In [8]:
# Task 1.1 — grain of raw.pos_transactions

# ==== YOUR ANSWER START ====
grain = q("""
select
  count(*) as n_rows,
  count(distinct(receipt_id)) as n_receipts,
  count(distinct((receipt_id, line_no))) as n_lines,
  (
    select max(n)
    from (
      select receipt_id, count(*) as n
      from raw.pos_transactions
      group by receipt_id
    )

  ) as max_lines_per_receipt
  from raw.pos_transactions
""")
# ==== YOUR ANSWER END ====
print(grain)

# --- Test
assert grain.n_rows[0] > grain.n_lines[0], "no duplicate line was detected"
assert grain.n_lines[0] > grain.n_receipts[0], "receipts don't have several lines"
assert grain.max_lines_per_receipt[0] > 1
print("Test 1.1 passed. One row of raw.pos_transactions is one product line of one receipt.")

   n_rows  n_receipts  n_lines  max_lines_per_receipt
0   14828        6634    14653                      6
Test 1.1 passed. One row of raw.pos_transactions is one product line of one receipt.


**Q1.** What is the grain of `raw.pos_transactions` ? What does the difference `n_rows - n_lines` measure? What does the difference `n_lines - n_receipts` measure? Which of the two points to a data quality defect, and which reflects a normal property of the application?

> 1. Each row is a transaction record. The intended grain of raw.pos_transactions is one product line within one receipt, uniquely identified by (receipt_id, line_no). `n_rows - n_lines` = extra rows caused by duplicated transaction lines.
> 2. one receipt can have multiple records. additional product lines because receipts can contain multiple items.
> 3.  n_rows > n_lines Therefore some (receipt_id, line_no) appears multiple times. So this indicates a data quality defect. n_lines > n_receipts is normal.

### Task 1.2 — NULL and rows excluded by a naive filter

**Course slides:** Section 1 - *SQL Three-Valued Logic*, Section 2 - *Silent Loss 2*.

**TO DO:** Write three queries on `pos_transactions`:
1. Count rows with `status` 'cancelled' and the number of rows with NULL values.
2. **Naive filter:** Count rows kept by `WHERE status <> 'cancelled'`.
3. Count rows kept by `WHERE status IS DISTINCT FROM 'cancelled'` (documentation link in the glossary above).

**Validation:** the naive filter should keep strictly fewer rows than the correct one.

In [9]:
# Task 1.2 — three-valued logic on a nullable flag

# ==== YOUR ANSWER START ====
per_status = q("""
select  COALESCE(status, '<NULL>') AS status_value, count(*) as n
from raw.pos_transactions
group by status
""")
n_naive   = con.execute("""
select count(*)
from raw.pos_transactions
WHERE status <> 'cancelled'
""").fetchone()[0]
n_correct = con.execute("""
select count(*)
from raw.pos_transactions
WHERE status IS DISTINCT FROM 'cancelled'
""").fetchone()[0]


# ==== YOUR ANSWER END ====
n_null = con.execute("SELECT count(*) FROM raw.pos_transactions WHERE status IS NULL").fetchone()[0]
print(per_status)
print("kept by  status <> 'cancelled'            :", n_naive)
print("kept by  status IS DISTINCT FROM 'cancelled':", n_correct)
print("rows with status IS NULL                  :", n_null)

# --- Test
assert n_naive < n_correct, "the naive filter did not lose anything, check the query"
assert n_correct - n_naive == n_null, "the difference must be exactly the NULL rows"
print("Test 1.2 passed. The naive filter discards", n_null, "valid sales.")

  status_value      n
0       <NULL>  13890
1    cancelled    938
kept by  status <> 'cancelled'            : 0
kept by  status IS DISTINCT FROM 'cancelled': 13890
rows with status IS NULL                  : 13890
Test 1.2 passed. The naive filter discards 13890 valid sales.


**Q2.** The naive filter runs without syntax errors or warnings. Using three-valued logic (`TRUE`, `FALSE`, `UNKNOWN`), explain which rows the query keeps or excludes. What underlying assumption does this filter make about records where the status is `NULL`?  *(Note: the meaning of NULL here depends on the design of the source system and can not be found by looking at the data alone. It must be confirmed with the application owner.)*

> 'cancelled' <> 'cancelled' = FALSE
> 'other' <> 'cancelled' = TRUE
> 'NULL' <> 'cancelled' = UNKNOWN
> the naive filter cannot select 'NULL' status since the condition is not TRUE


### Task 1.3 — Foreign Keys with no reference

**Course slides:** Section 2 - *Silent Loss 1, INNER JOIN*, Section 3 - *Form 3, detecting rows dropped by a join*.

**TO DO:** Write two anti-joins (`LEFT JOIN` followed by `WHERE <right key> IS NULL`), each returning the row count and the total revenue of the matching rows (`sum(amount_cents) / 100`):

1. Rows in `raw.pos_transactions` whose `store_code` has no match in `raw.stores`.
2. Rows in `raw.pos_transactions` whose `crm_ref` has no match in `raw.crm_customers`.

Foreign keys with no matching reference are not data errors. They can reflect real situations: a store closing down or a new store opening before the reference file is updated, or a sale happening without a loyalty card being scanned. These rows must be kept and counted in total revenue, since they represent sales that took place and were paid for.

**Validation:** both anti-joins return a non-zero number of rows.


In [10]:
# Task 1.3 — foreign keys with no matching reference row
# ==== YOUR ANSWER START ====
orphan_stores = q("""
select count(*) as n_rows, sum(a.amount_cents) / 100 as totalrevenue
from raw.pos_transactions a left join raw.stores b on a.store_code = b.store_code
WHERE b.store_code IS NULL
""")
orphan_customers = q("""
select count(*) as n_rows, sum(a.amount_cents) / 100 as totalrevenue
from raw.pos_transactions a left join raw.crm_customers b on a.crm_ref = b.crm_ref
WHERE b.crm_ref IS NULL
""")
# ==== YOUR ANSWER END ====

print("store codes absent from the the reference store table:"); print(orphan_stores)
print("customer references absent from the teh reference customer table:"); print(orphan_customers)

# --- Test
assert len(orphan_stores) > 0 and orphan_stores.n_rows.sum() > 0
assert orphan_customers.n_rows[0] > 0
print("Test 1.3 passed. An INNER JOIN on either key would silently delete these rows.")

store codes absent from the the reference store table:
   n_rows  totalrevenue
0     627      29696.23
customer references absent from the teh reference customer table:
   n_rows  totalrevenue
0     447      27808.18
Test 1.3 passed. An INNER JOIN on either key would silently delete these rows.


# Part 2 — Building the dimensions

**Steps 2,3,4:**
**Course slides:** Section 1 - *Handling the Unknown member*, Section 4 - *SCD Type 2* and *Validation: Grain Uniqueness and Temporal Intervals*, Section 3 - *Semi-Additive Measures*.

**Input:** `raw.stores`, `raw.products`, `raw.crm_customers`, `raw.crm_customers_d2`, `raw.daily_inventory`. `mart.dim_date` and `mart.dim_product` are provided.

**Output:**
- `mart.dim_date` (provided): one row per calendar day, covering a period wide enough for the scoring date used in Part 5.
- `mart.dim_product` (provided): one row per product, plus an Unknown row, built directly from `raw.products`.
- `mart.dim_store`: one row per store, plus an Unknown row (`store_sk = -1`) and a row for the online channel, built directly from `raw.stores`.
- `mart.dim_customer_scd2`: one row per customer state, plus an Unknown row, built through three intermediate staging tables:
  - `stg.crm_v1`, `stg.crm_v2`: the two CRM snapshots, cleaned with the same rules, so they can be compared directly.
  - `stg.scd_changed`: the customers whose city or segment differs between the two snapshots, used to decide which customers get a second row.
  `mart.dim_customer_scd2` is then assembled from `stg.crm_v1`, `stg.crm_v2`, and `stg.scd_changed`, giving each customer one row per state, with a validity interval so a sale can later be matched to the customer as they were at the time of that sale.
- Validation results confirming that these validity intervals have no overlap, no gap, and exactly one current version per customer.

Each dimension table has an explicit Unknown row (`store_sk = -1`, `customer_sk = -1`). Sales with a missing foreign key will be matched to this row, so their revenue is preserved in the total.



### Cell 2.0 — `mart.dim_date` and `mart.dim_product` (Provided)

`mart.dim_product` is the product dimension, one row per article sold by MegaStore, built from `raw.products`. The line corresponfing to the Unknown member is inserted.

`mart.dim_date` spans from 2024-01-01 through 2026-06-30. Sales records only run from 2024-01-01 to 2025-12-31, but the calendar extends past the final purchase so later scoring dates still match a valid record in `mart.dim_date`.

In [11]:
# Cell 2.0 — calendar and product dimensions
con.execute("""
CREATE OR REPLACE TABLE mart.dim_date AS
SELECT row_number() OVER (ORDER BY d)   AS date_sk,
       d                                AS date_day,
       extract('year'  FROM d)          AS year_num,
       extract('month' FROM d)          AS month_num,
       strftime(d, '%Y-%m')             AS year_month,
       (extract('dow' FROM d) IN (0, 6)) AS is_weekend
FROM (SELECT unnest(generate_series(DATE '2024-01-01', DATE '2026-06-30', INTERVAL 1 DAY)) AS d)
""")

# Build the product dimension from raw.products, with a surrogate key and cleaned labels
con.execute("""
CREATE OR REPLACE TABLE mart.dim_product AS
SELECT row_number() OVER (ORDER BY barcode) AS product_sk,
       barcode, clean_text(product_name) AS product_name, brand, category, unit_cost, unit_price
FROM raw.products
""")
# Add the Unknown member, product_sk = -1
con.execute("INSERT INTO mart.dim_product VALUES (-1, 'UNKNOWN', 'Unknown Product', 'Unknown', 'Unknown', 0.0, 0.0)")


display(q("SELECT * FROM mart.dim_product ORDER BY product_sk LIMIT 4").style.hide(axis="index"))

display(q("SELECT * FROM mart.dim_date LIMIT 4").style.hide(axis="index"))

product_sk,barcode,product_name,brand,category,unit_cost,unit_price
-1,UNKNOWN,Unknown Product,Unknown,Unknown,0.000000,0.000000
1,3000000000052,still water 1l,Cristalline,Beverages,0.400000,1.100000
2,3000000000113,ORANGE JUICE 1L,Tropicana,Beverages,0.900000,2.300000
3,3000000000133,COLA SODA 1.5L,CocaCola,Beverages,0.700000,1.800000


date_sk,date_day,year_num,month_num,year_month,is_weekend
1,2024-01-01 00:00:00,2024,1,2024-01,False
2,2024-01-02 00:00:00,2024,1,2024-01,False
3,2024-01-03 00:00:00,2024,1,2024-01,False
4,2024-01-04 00:00:00,2024,1,2024-01,False


### Task 2.1 — Build `mart.dim_store`

**Course slides:** Section 1 - *Handling Missing Keys: the Unknown member*.

**TO DO:** Build `mart.dim_store` in three statements:
1. `CREATE OR REPLACE TABLE` from `raw.stores`, adding a surrogate key `store_sk` with `row_number() OVER`.
2. `INSERT` one row for the online store: `store_code = 'WEB'`, `region = 'Digital'`. This row has no that has no corresponding entry in `raw.stores`

3. `INSERT` the Unknown member: `store_sk = -1`, `store_code = 'UNKNOWN'`, every other column set to `'Unknown'` (In Part 3, any sale recorded under the unregistered store code `M99` is matched to this row).

**Validation:** the surrogate key is unique, the row `-1` exists, the row `WEB` exists in `mart.dim_store`.

In [12]:
# Task 2.1 — store dimension

# ==== YOUR ANSWER START  ====
con.execute("""
CREATE OR REPLACE TABLE mart.dim_store AS
SELECT row_number() OVER (ORDER BY store_code) AS store_sk,
       store_code, clean_text(store_name) AS store_name, city, region
FROM raw.stores
""")

con.execute("""
INSERT INTO mart.dim_store
  select MAX(store_sk) + 1, 'WEB', 'Web store','Unknown', 'Digital'
  from  mart.dim_store
""")

con.execute("""
INSERT INTO mart.dim_store
  VALUES (-1, 'UNKNOWN', 'Unknown', 'Unknown', 'Unknown')
""")

# ==== YOUR ANSWER END ====

dim_store = q("SELECT * FROM mart.dim_store ORDER BY store_sk")
display(dim_store.style.hide(axis="index"))

# --- Test
assert dim_store.store_sk.is_unique, "store_sk is not unique"
assert (dim_store.store_sk == -1).any(), "the Unknown member is missing"
assert (dim_store.store_code == 'WEB').any(), "the online channel row is missing"
print("Test 2.1 passed.")

store_sk,store_code,store_name,city,region
-1,UNKNOWN,Unknown,Unknown,Unknown
1,M01,MegaStore Strasbourg,Strasbourg,Grand Est
2,M02,MegaStore Paris Bastille,Paris,Ile-de-France
3,M03,MegaStore Lyon Part-Dieu,Lyon,Auvergne-Rhone-Alpes
4,M04,MegaStore Toulouse,Toulouse,Occitanie
5,M05,MegaStore Lille,Lille,Hauts-de-France
6,M06,MegaStore Nantes,Nantes,Pays de la Loire
7,WEB,Web store,Unknown,Digital


Test 2.1 passed.


### Task 2.2 — Versioned customer dimension: `mart.dim_customer_scd2`

**Course slides:** Section 4 - *SCD Type 2: Preserving History with Versioned Rows*.

**Input:** `raw.crm_customers` (state known from 2024-01-01) and `raw.crm_customers_d2` (state known from 2025-06-01).

**Output:** one row per **state** of a customer, with a half-open validity interval `[valid_from, valid_to[`.


**Validation:** `n_rows = n_customers + n_changed + 1`, where:

* `n_changed` : number of customers whose state changed between the two snapshots
* `n_customers` : number of customers in each snapshot (300 in each one)
* `n_rows` : number of rows in `mart.dim_customer_scd2`
* 1: default sentinel row (Unknown member)

The open end is written `9999-12-31`, so that a date comparison doesn't have to handle NULL.

There are four sub-tasks.

**Task 2.2a — normalise the first snapshot.**

**TO DO:** Create the view `stg.crm_v1` over `raw.crm_customers` with the columns `crm_ref`, `full_name`, `gender`, `city`, `segment`, applying:
- `clean_text(full_name)` to remove leading, trailing and repeated spaces,
- a `CASE` mapping `gender` to `'M'`, `'F'`, or `'Unknown'`. Values observed in the source are `M`, `F`, `m`, `f`, `1`, `2`. Any other value should be assigned to `'Unknown'`
- `trim(split_part(city, '(', 1))` to drop the department code appended to some city names.

This view normalises the first CRM snapshot so it can later be compared to the second one, and eventually assembled into `mart.dim_customer_scd2`.


`stg.crm_v1` is a view because it only applies a light transformation, read once afterwards, and does not need to be stored (same for `stg.crm_v2`).

In [13]:
# Task 2.2a — normalised first snapshot

# ==== YOUR ANSWER START ====
con.execute("""
CREATE OR REPLACE VIEW stg.crm_v1 AS (
  select crm_ref, clean_text(full_name) as full_name,
  case when gender in ('M','m','1') then 'M'
    when gender in ('F','f','2') then 'F'
    else 'Unknown' end as gender, trim(split_part(city, '(', 1)) as city, segment
  from raw.crm_customers
)
""")
# ==== YOUR ANSWER END ====

display(q("SELECT * FROM stg.crm_v1 LIMIT 4").style.hide(axis="index"))

assert con.execute("SELECT count(*) FROM stg.crm_v1 WHERE gender NOT IN ('M','F','Unknown')").fetchone()[0] == 0
print("Test 2.2a passed.")

crm_ref,full_name,gender,city,segment
CLI-4001,Fontaine Fatima,M,Metz,Pro
CLI-4002,PETIT c.,F,Lyon,VIP
CLI-4003,chloe FAURE,M,Bordeaux,VIP
CLI-4004,Paul Martin,F,Metz,Pro


Test 2.2a passed.


**Task 2.2b — normalise the second snapshot.**  

**TO DO:**  Build `stg.crm_v2` by applying the same transformations over `raw.crm_customers_d2`. The two views must have identical column names and identical cleaning rules.

In [14]:
# Task 2.2b — normalised second snapshot

# ==== YOUR ANSWER START ====
con.execute("""
CREATE OR REPLACE VIEW stg.crm_v2 AS (
  select crm_ref, clean_text(full_name) as full_name,
  case when gender in ('M','m','1') then 'M'
    when gender in ('F','f','2') then 'F'
    else 'Unknown' end as gender, trim(split_part(city, '(', 1)) as city, segment
  from raw.crm_customers_d2
)
""")
# ==== YOUR ANSWER END ====

assert set(q("SELECT * FROM stg.crm_v1 LIMIT 0").columns) == set(q("SELECT * FROM stg.crm_v2 LIMIT 0").columns)
print("Test 2.2b passed. The two snapshots are comparable.")

Test 2.2b passed. The two snapshots are comparable.


**Task 2.2c — identify the customers whose state changed.**

**TO DO:** Use the two previous views to create the table `stg.scd_changed` holding one column `crm_ref`, containing the customers for which `city` **or** `segment` differs between the two snapshots. Use `IS DISTINCT FROM` instead of  `<>` (see Task1.2).

`stg.scd_changed` is stored as a table because it is read more than once while assembling `mart.dim_customer_scd2`, thus avoiding recomputing the comparison of the two snapshots each time it is referenced.

In [15]:
# Task 2.2c — customers whose tracked attributes changed

# crm_v1 = customer state in 2024
# crm_v2 = customer state in 2025
# ==== YOUR ANSWER START  ====
con.execute("""
CREATE OR REPLACE TABLE stg.scd_changed AS (
  select distinct a.crm_ref
  from stg.crm_v1 a join stg.crm_v2 b on a.crm_ref = b.crm_ref
  where a.city IS DISTINCT FROM b. city or a.segment IS DISTINCT FROM b. segment
)
""")
# ==== YOUR ANSWER END ====

n_changed = con.execute("SELECT count(*) FROM stg.scd_changed").fetchone()[0]
print("customers with a changed state:", n_changed)
assert n_changed > 0
print("Test 2.2c passed.")

customers with a changed state: 37
Test 2.2c passed.


**Task 2.2d — assemble the dimension.**

**TO DO:** Create `mart.dim_customer_scd2` with the columns
`customer_sk, crm_ref, full_name, gender, city, segment, valid_from, valid_to, is_current`:

* **Base version (Snapshot 1):** Include every customer with `valid_from = '2024-01-01'`. Set `valid_to = '2025-06-01'` if the customer appears in `stg.scd_changed`, and `'9999-12-31'` otherwise (you can use `CASE`-`WHEN`-`ELSE`).
* **Updated version (Snapshot 2):** Append a second row for customers in `stg.scd_changed` with `valid_from = '2025-06-01'` and `valid_to = '9999-12-31'`.
* **Surrogate key:** Generate `customer_sk` with `ROW_NUMBER() OVER (ORDER BY crm_ref, valid_from)`.
* **Current record flag:** Set `is_current = TRUE` when `valid_to = DATE '9999-12-31'`, and `FALSE` otherwise.
* **Unknown member:** Append the default sentinel record with `customer_sk = -1`, `crm_ref = 'UNKNOWN'`, and an interval spanning `[1900-01-01, 9999-12-31[`.

In [16]:
# Task 2.2d — versioned customer dimension
# ==== YOUR ANSWER START  ====
con.execute("""
CREATE OR REPLACE TABLE mart.dim_customer_scd2 AS
WITH versions AS (
    -- Snapshot 1
    SELECT a.crm_ref, full_name, gender, city, segment,  DATE '2024-01-01' as valid_from, --debug date
           CASE WHEN c.crm_ref IS NOT NULL THEN  DATE '2025-06-01' else '9999-12-31' end as valid_to
    from stg.crm_v1 a left join stg.scd_changed c on a.crm_ref = c.crm_ref

    UNION ALL

    -- Snapshot 2
    SELECT a.crm_ref, full_name, gender, city, segment, DATE '2025-06-01' as valid_from,
       DATE '9999-12-31' as valid_to
    from stg.crm_v2 a join stg.scd_changed c on a.crm_ref = c.crm_ref
)
-- surrogate key and current flag
SELECT ROW_NUMBER() OVER (ORDER BY crm_ref, valid_from) AS customer_sk, -- build the surrogate key
       *,(valid_to = DATE '9999-12-31') AS is_current
FROM versions
""")

#Add the Unknown member, customer_sk = -1
con.execute("""
INSERT INTO mart.dim_customer_scd2
  VALUES (-1, 'UNKNOWN', 'Unknown', 'Unknown', 'Unknown', 'Unknown','1900-01-01', '9999-12-31',TRUE)
""")
# ==== YOUR ANSWER END ====

display(q("""SELECT customer_sk, crm_ref, city, segment, valid_from, valid_to, is_current
           FROM mart.dim_customer_scd2
           WHERE crm_ref IN (SELECT crm_ref FROM stg.scd_changed LIMIT 2)
           ORDER BY crm_ref, valid_from"""))

# --- Test
n_customers = con.execute("SELECT count(*) FROM stg.crm_v1").fetchone()[0]
n_changed   = con.execute("SELECT count(*) FROM stg.scd_changed").fetchone()[0]
n_rows       = con.execute("SELECT count(*) FROM mart.dim_customer_scd2").fetchone()[0]
display(f"customers {n_customers}, changed {n_changed}, dimension rows {n_rows}")
assert n_rows == n_customers + n_changed + 1, "row count does not match the expected number of states"
assert con.execute("SELECT count(*) FROM mart.dim_customer_scd2 WHERE customer_sk = -1").fetchone()[0] == 1
print("Test 2.2d passed.")

,customer_sk,crm_ref,city,segment,valid_from,valid_to,is_current
0,11,CLI-4011,Lyon,VIP,2024-01-01,2025-06-01,False
1,12,CLI-4011,Toulouse,VIP,2025-06-01,9999-12-31,True
2,16,CLI-4015,Strasbourg,Pro,2024-01-01,2025-06-01,False
3,17,CLI-4015,Paris,VIP,2025-06-01,9999-12-31,True


'customers 300, changed 37, dimension rows 338'

Test 2.2d passed.


### Task 2.3 — Validate `mart.dim_customer_scd2`

**Course slide:** Section 4 - *Validation: Grain Uniqueness and Temporal Intervals*.

Point-in-time joins rely on three constraints that must be tested explicitly, as the database does not enforce them automatically.


**TO DO:** Write one query per check, each returning a single count:

1. **No overlapping intervals:** Validity periods for the same customer must not intersect (`a.valid_from < b.valid_from AND a.valid_from < b.valid_to AND b.valid_from < a.valid_to`). Overlaps cause as-of joins to match multiple records, silently duplicating fact rows. Test this constraint using a self join on `crm_ref`.
2. **No timeline gaps:** For consecutive versions of a customer ordered by `valid_from`, the next `LEAD(valid_from)` must match the current `valid_to`. A gap causes an as-of join to drop the observation rows.
3. **Exactly one active version:** Exactly one row with `is_current = TRUE` per `crm_ref`. Zero or multiple active rows is ambiguous.

**Validation:** the three checks return 0.

In [17]:
# Task 2.3 — three validity checks on the versioned dimension

# ==== YOUR ANSWER START  ====
n_overlap = con.execute("""
SELECT count(*)
from mart.dim_customer_scd2 a join mart.dim_customer_scd2 b on a.crm_ref = b.crm_ref
where a.valid_from < b.valid_from AND a.valid_from < b.valid_to AND b.valid_from < a.valid_to
""").fetchone()[0]

n_gap = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT
        crm_ref,
        valid_to,
        LEAD(valid_from) OVER (
            PARTITION BY crm_ref
            ORDER BY valid_from
        ) AS next_valid_from
    FROM mart.dim_customer_scd2
)
WHERE next_valid_from IS NOT NULL --debug
  AND valid_to IS DISTINCT FROM next_valid_from --debug
""").fetchone()[0]

n_bad_current = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT
        crm_ref,
        SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS n_current --debug
    FROM mart.dim_customer_scd2
    GROUP BY crm_ref
)
WHERE n_current <> 1 --debug not >1
""").fetchone()[0]
# ==== YOUR ANSWER END ====

print("overlapping pairs:", n_overlap, "| gaps:", n_gap, "| customers without exactly one current row:", n_bad_current)

# --- Test
assert n_overlap == 0, "overlapping intervals, the as-of join would duplicate rows"
assert n_gap == 0, "gap between versions, the as-of join would drop rows"
assert n_bad_current == 0
print("Test 2.3 passed. The dimension can be joined as of a date.")

overlapping pairs: 0 | gaps: 0 | customers without exactly one current row: 0
Test 2.3 passed. The dimension can be joined as of a date.


### Cell 2.4 — Example of a non-additive measure (Provided)

**Course frame:** Section 3 - *Form 1, Measure Additivity* and *Semi-Additive Measures*.

`raw.daily_inventory` records the stock held in a store, for a product, at the end of each day. The cell below computes three aggregates over one store, one product, over seven days. Read the three numbers before answering.

In [18]:
# Cell 2.4 — three aggregates on a stock level
display(q(sql="""
SELECT store_code, barcode,
       sum(stock_units)                     AS sum_over_7_days,
       round(avg(stock_units), 1)           AS avg_daily_stock,
       arg_max(stock_units, snapshot_date)  AS closing_stock
FROM raw.daily_inventory
WHERE store_code = 'M01'
  AND barcode = (SELECT min(barcode) FROM raw.daily_inventory)
  AND snapshot_date BETWEEN DATE '2024-01-01' AND DATE '2024-01-07'
GROUP BY store_code, barcode
"""))

,store_code,barcode,sum_over_7_days,avg_daily_stock,closing_stock
0,M01,3000000000052,516.0,73.7,33


**Q3.** Which of the three numbers answers the question *how many units were held on 7 January*, and which one answers *what was the typical stock level over the week*? Explain what `sum_over_7_days` actually counts. Name one measure in `raw.pos_transactions` that can be summed across time without issue and explain why the column type alone is not an indicator for this difference.

> closing_stock //
> avg //
> sum of every day's inventory //
> amount_cents, quantity //
closing_stock gives the number of units held on 7 January, because arg_max(stock_units, snapshot_date) returns the stock value associated with the latest date. avg_daily_stock represents the typical stock level during the week. sum_over_7_days sums the seven daily inventory snapshots; it does not represent a physical number of units held, but rather a sum across repeated stock observations (roughly unit-days). In raw.pos_transactions, amount_cents can be summed across time to calculate total sales because each row represents a transaction event. Whether a measure is additive depends on its business meaning, not its SQL data type: both inventory levels and sales amounts may be numeric, but inventory is a snapshot while sales are flows.

# Part 3 — Bulding the fact table `mart.fact_sales`

**Steps 5-8:**
**Course slides:**  *Transformation Logic*, Section 1 - *Fact Tables* and *Unknown member*, Section 4 - *Point-in-Time (As-Of) Join*, Section 3 - *Form 2, Join Fan-Out* and *Aggregate to Target Grain Before Joining*.

**Input:** `raw.pos_transactions`, `raw.web_orders`, the four dimensions seen previously.

**Output:** `mart.fact_sales`, built by using three intermediate staging tables:
- `stg.sales`: the in-store transactions, de-duplicated and converted to a common unit, one row per product line.
- `stg.web_orders`: the online orders, restructured into the same shape as `stg.sales`.
- `stg.sales_all`: `stg.sales` and `stg.web_orders` combined into a single set of sales lines


`mart.fact_sales` is then assembled from `stg.sales_all` by resolving every store, product, and customer reference against the dimensions from Part 2, including the references that could not be matched.

### Cell 3.0 — `stg.web_orders` (Provided)

The web data is already clean, it only needs restructuring: extracting the nested `device` field from the JSON object and filtering out cancelled orders using the predicate validated in Task 1.2.


In [19]:
# Cell 3.0 — web sales staging
con.execute("""
CREATE OR REPLACE TABLE stg.web_orders AS
SELECT order_id            AS receipt_id,
       line_no,
       order_date          AS sale_date,
       sku                 AS barcode,
       'WEB'               AS store_code,
       customer_ref        AS crm_ref,
       qty                 AS quantity,
       amount_eur          AS amount_net,
       metadata.device     AS device
FROM raw.web_orders
WHERE status IS DISTINCT FROM 'cancelled'
""")

display(q("SELECT * FROM stg.web_orders LIMIT 3"))



,receipt_id,line_no,sale_date,barcode,store_code,crm_ref,quantity,amount_net,device
0,900001,1,2024-01-03,3000000000967,WEB,CLI-4192,1,2.2,mobile_android
1,900001,2,2024-01-03,3000000000113,WEB,CLI-4192,1,2.3,mobile_ios
2,900001,3,2024-01-03,3000000000265,WEB,CLI-4192,1,2.1,mobile_android


### Task 3.1a — `stg.sales` and `stg.sales_all`

**Course slide:** *Transformation Logic: From Raw Sources to Warehouse Tables*.


Create `stg.sales` from `raw.pos_transactions` with the columns `receipt_id`, `line_no`, `sale_date`, `barcode`, `store_code`, `crm_ref`, `quantity`, and `amount_net`.

Apply five transformation rules:

| Rule | Rationale |
|---|---|
| Deduplicate on `(receipt_id, line_no)` | Remove exact duplicate lines identified in Task 1.1 |
| `amount_net = amount_cents / 100.0` | Convert in-store cents to euros to match web orders |
| `quantity > 0` AND `amount_cents IS NOT NULL` | Drop invalid lines with negative counts or missing values |
| `amount_cents <= 100000` | Remove cash register errors above 1,000 euros (audited in Task 3.1c) |
| Filter out cancelled sales | Apply the cancellation predicate validated in Task 1.2 |

The 1,000-euro cap is a specific rule for this lab, not a general standard. Any item line over 1,000 euros is treated as a cash register typing error, not a real sale. Because dropping these lines changes total revenue, Task 3.1c tracks and displays the removed money so the change stays visible.

Then create the view `stg.sales_all` as the `UNION ALL` of `stg.sales` and `stg.web_orders`, restricted to the eight common columns.

**Validation:** the pair `(receipt_id, line_no)` is unique, there is no NULL amount, the quantity is greater than 1, there is no line above 1000 euros.

In [20]:
# Task 3.1a
# ==== YOUR ANSWER START  ====
con.execute("""
CREATE OR REPLACE TABLE stg.sales AS
SELECT receipt_id, line_no, sale_date, barcode, store_code, crm_ref, quantity,
       amount_cents / 100.0 AS amount_net
FROM (
    SELECT *, row_number() over (PARTITION BY receipt_id, line_no) as rn
    from raw.pos_transactions
) t
WHERE rn = 1
  AND quantity > 0
  AND amount_cents IS NOT NULL
  AND amount_cents <= 100000
  AND status IS DISTINCT FROM 'cancelled'
""")

con.execute("""
CREATE OR REPLACE VIEW stg.sales_all AS
select * from stg.sales
UNION ALL
select receipt_id, line_no, sale_date, barcode, store_code, crm_ref, quantity, amount_net
from stg.web_orders
""")
# ==== YOUR ANSWER END ====

chk = q("""
SELECT count(*) AS n_rows, count(DISTINCT (receipt_id, line_no)) AS n_keys,
       min(quantity) AS min_quantity, max(amount_net) AS max_line_amount,
       count(*) FILTER (WHERE amount_net IS NULL) AS n_null_amount
FROM stg.sales_all
""")
display(chk)

# --- Test
assert chk.n_rows[0] == chk.n_keys[0], "duplicate lines remain"
assert chk.min_quantity[0] >= 1 and chk.n_null_amount[0] == 0
assert chk.max_line_amount[0] <= 1000, "an incorrect line amount remains"
print("Test 3.1a passed.")

,n_rows,n_keys,min_quantity,max_line_amount,n_null_amount
0,15713,15713,1,39.0,0


Test 3.1a passed.


### Task 3.1b — `mart.fact_sales`

 Build the final fact table `mart.fact_sales` from `stg.sales_all` by linking every transaction to its dimension keys, without losing rows or altering total revenue.

**Course slides:** Section 1 - *Unknown member*, Section 4 - *The Solution: Point-in-Time (As-Of) Join*.

**Problem:** Standard `INNER JOIN` operations silently delete sales rows whenever a store, product, or customer key is missing. Furthermore, customer data changes over time, so sales must be linked to the customer's profile as it existed on the day of the purchase.

**Instructions:** Create `mart.fact_sales` from `stg.sales_all` using four `LEFT JOIN` clauses:

| Dimension | Join Condition | Value if Unmatched |
|---|---|---|
| `mart.dim_date` | `sale_date = date_day` | not a possible outcome (no such defect was injected in this dataset) |
| `mart.dim_product` | `barcode = barcode` | `-1` using `COALESCE` |
| `mart.dim_store` | `store_code = store_code` | `-1` using `COALESCE` |
| `mart.dim_customer_scd2` | `crm_ref = crm_ref` AND `sale_date >= valid_from` AND `sale_date < valid_to` | `-1` using `COALESCE` |

* **Point-in-time join:** The date conditions on customer history ensure that this join returns exactly one row per sale line.
* **Two date columns:** Keep both `sale_date` (used by the database engine to skip data blocks and speed up queries) and `date_sk` (used to join the calendar and group by month or week).

**Validation:** The test checks that:
* Total row count matches `stg.sales_all` exactly (no rows dropped or duplicated).
* Total revenue matches `stg.sales_all` exactly.
* `date_sk` contains zero NULL values across the entire table, since the calendar is expected to cover the entire sales period without gaps.

In [21]:
# Task 3.1b — fact table with resolved keys
# ==== YOUR ANSWER START  ====
con.execute("""
CREATE OR REPLACE TABLE mart.fact_sales AS
SELECT date_sk, v.*,
      coalesce(p.product_sk, -1) as product_sk,
      COALESCE(b.store_sk, -1) AS store_sk, -- debug
      COALESCE(c.customer_sk, -1) AS customer_sk -- debug
FROM stg.sales_all v
LEFT JOIN mart.dim_product p on v.barcode = p.barcode
LEFT JOIN mart.dim_date a on v.sale_date = a.date_day
LEFT JOIN mart.dim_store b on v.store_code = b.store_code
LEFT JOIN mart.dim_customer_scd2 c on v.crm_ref = c.crm_ref
      AND v.sale_date >= c.valid_from
      AND v.sale_date <  c.valid_to
""")
# ==== YOUR ANSWER END ====
rec = q("""
SELECT (SELECT count(*) FROM stg.sales_all)                     AS source_rows,
       (SELECT count(*) FROM mart.fact_sales)                   AS fact_rows,
       (SELECT round(sum(amount_net), 2) FROM stg.sales_all)    AS source_revenue,
       (SELECT round(sum(amount_net), 2) FROM mart.fact_sales)  AS fact_revenue,
       (SELECT count(*) FROM mart.fact_sales WHERE date_sk IS NULL) AS unresolved_dates
""")
display(rec)

# --- Test
assert rec.source_rows[0] == rec.fact_rows[0], "the fact table lost or duplicated rows"
assert rec.source_revenue[0] == rec.fact_revenue[0], "revenue is not preserved"
assert rec.unresolved_dates[0] == 0, "a sale date is missing from the calendar"
print("Test 3.1b passed. No row lost, no row duplicated, revenue preserved.")

,source_rows,fact_rows,source_revenue,fact_revenue,unresolved_dates
0,15713,15713,73695.1,73695.1,0


Test 3.1b passed. No row lost, no row duplicated, revenue preserved.


### Cell 3.1 — Data Quality of `mart.fact_sales` (Provided)

Create a single summary table that reports both the row count and the total revenue corresponding to each data quality issue identified so far in `mart.fact_sales`.

The summary has three columns: `bucket`, `n_rows`, and `revenue`:
* `'store'`: sales with an unresolved store (`store_sk = -1` in `mart.fact_sales`, sum of `amount_net`)
* `'product'`: sales with an unresolved product (`product_sk = -1` in `mart.fact_sales`, sum of `amount_net`)
* `'customer'`: sales with an unresolved customer (`customer_sk = -1` in `mart.fact_sales`, sum of `amount_net`)
* `'excluded_outlier'`: sales above 1000 euros excluded from the cash register file (`amount_cents > 100000` in `raw.pos_transactions`, sum of `amount_cents / 100.0`)



In [22]:
# Task 3.1c — revenue report
report = q("""
SELECT 'store'    AS bucket, count(*) AS n_rows, round(sum(amount_net), 2) AS revenue
FROM mart.fact_sales WHERE store_sk = -1
UNION ALL
SELECT 'product', count(*), round(sum(amount_net), 2) FROM mart.fact_sales WHERE product_sk = -1
UNION ALL
SELECT 'customer', count(*), round(sum(amount_net), 2) FROM mart.fact_sales WHERE customer_sk = -1
UNION ALL
SELECT 'excluded_outlier', count(*), round(sum(amount_cents) / 100.0, 2)
FROM raw.pos_transactions WHERE amount_cents > 100000
""")

display(report)

,bucket,n_rows,revenue
0,store,561,2856.30
1,product,275,2715.00
2,customer,398,1885.20
3,excluded_outlier,89,441653.76


**Q4.** Compare three ways to handle the invalid `M99` lines: deleting them, setting `store_sk` to NULL, or mapping them to `store_sk = -1`. For each option, explain what happens to total revenue and to reports built on `mart.dim_store` (such as revenue grouped by `region`). Then explain why sales with `customer_sk = -1` are kept in the fact table but must be excluded from the training data.

> deleting them would lose the revenue; to null dont lose the record but not showing in the report.
>
Deleting the invalid `M99` sales would remove those rows from the fact table, so their revenue would be lost and total revenue would be understated.

Setting `store_sk` to `NULL` would preserve the sales and therefore preserve total revenue when querying the fact table directly. However, reports that join `mart.fact_sales` to `mart.dim_store`, such as revenue grouped by `region`, may exclude these rows because a NULL key cannot match any row in the store dimension. With a left join they could appear under a NULL category, but they still have no proper dimensional member.

Mapping the invalid stores to `store_sk = -1` is preferable. The sales and their revenue are preserved, and `-1` joins to the explicit `Unknown` row in `mart.dim_store`. Therefore dimensional reports can include the revenue under an `Unknown` region/store rather than silently dropping it.

Sales with `customer_sk = -1` are also kept in the fact table because they are real sales and must contribute to revenue and other business totals. However, they should be excluded from customer-level training data because the customer identity is unknown. Treating all unknown customers as customer `-1` would incorrectly combine purchases from many unrelated people into one artificial customer, producing meaningless features such as purchase frequency, recency, and average spend.

### Task 3.2 — Example of Join Fan-Out Between Two Fact Tables

Demonstrate how joining two fact tables directly duplicates rows (fan-out) and inflates numbers, then fix it by aggregating before joining.

**Course slides:** Section 3 - *Form 2, Join Fan-Out* and *Method: Aggregate to Target Grain Before Joining*

**Problem:** `raw.support_tickets` records one row per support ticket. If you join it directly to `mart.fact_sales` on `crm_ref`, a customer with $N$ sales and $M$ tickets produces $N \times M$ rows, artificially multiplying the revenue.

**Instructions:**
1. **Select a test customer:** Use the provided query to pick a customer with at least 4 tickets.
2. **Run the naive query:** Join `mart.fact_sales` directly to `raw.support_tickets` on `crm_ref` and calculate `sum(amount_net)` for that customer.
3. **Run the correct query:** Aggregate each table by customer in separate CTEs first, then join the aggregated results.
4. **Compare:** Print both revenue totals and their ratio (the inflation factor).

**Validation:** The test checks that the naive query inflates the calculated revenue by a factor strictly greater than 2.

In [23]:
# Task 3.2 — fan-out between two fact tables
test_customer = con.execute("""
SELECT t.crm_ref FROM raw.support_tickets t
JOIN (SELECT DISTINCT crm_ref FROM mart.fact_sales) f USING (crm_ref)
GROUP BY t.crm_ref HAVING count(*) >= 4
ORDER BY count(*) DESC LIMIT 1
""").fetchone()[0]
print("test customer:", test_customer)

# ==== YOUR ANSWER START  ====
naive = con.execute(f"""
SELECT sum(amount_net) as revenue
from mart.fact_sales f join raw.support_tickets b USING (crm_ref)
WHERE f.crm_ref = '{test_customer}'
""").fetchone()[0]

correct = con.execute(f"""
WITH sales_by_customer AS (
    select crm_ref, sum(amount_net) as revenue
    from mart.fact_sales
    group by crm_ref
),
tickets_by_customer AS (
    select crm_ref,count(*) as n
    from raw.support_tickets
    group by crm_ref
)
SELECT round(s.revenue, 2)
FROM sales_by_customer s
LEFT JOIN tickets_by_customer t USING (crm_ref)
WHERE s.crm_ref = '{test_customer}'
""").fetchone()[0]
# ==== YOUR ANSWER END ====

n_tickets = con.execute(f"SELECT count(*) FROM raw.support_tickets WHERE crm_ref = '{test_customer}'").fetchone()[0]
print(f"naive revenue   : {naive}")
print(f"correct revenue : {correct}")
print(f"inflation factor: {naive / correct:.1f}x   (tickets for this customer: {n_tickets})")

# --- Validation
assert naive / correct > 2, "no inflation observed, check the naive query"
assert abs(naive / correct - n_tickets) < 1e-6, "the factor should equal the number of tickets"
print("Test 3.2 passed.")

test customer: CLI-4126
naive revenue   : 3445.000000000001
correct revenue : 344.5
inflation factor: 10.0x   (tickets for this customer: 10)
Test 3.2 passed.


**Q5.** Consider a customer `i` with $N_i$ sales lines and $M_i$ support tickets. How many rows does a direct join produce for this customer? By what factor is `sum(amount_net)` inflated? Why is it impossible to fix this error using a single constant after aggregating? What happens to `count(t.ticket_id)` in the same join?

> N*M
> and revenue is inflated by:
\[
M_i
\]
because every sales line appears once for each ticket. count(ticket_id) is inflated by \(N_i\), because every ticket appears once for every sales line.
>

# Part 4 — Building `mart.training_churn`

**Steps 9–12.**
**Course slides:** Section 5 — *Building the Cohort*, *Dynamic Feature Lookback Windows*, *Lookback Window Aggregation*, *Sequential Features: LAG*, *Target Label Construction*, *Assembling the Labelled Table*, *Temporal Train-Test Split*, *Testing Temporal Boundaries*.


Prediction goal: **will this customer buy again within the next 60 days?**

One row of the training table is not a customer, it is a customer **observed at a date**. The same customer observed at two different time periods gives two rows, with different features and possibly different labels.

**Timeline and Observation Windows:**

* **Features (365-day lookback):** Aggregates customer purchase history ending 2 days before the observation date (`sale_date < obs_date - 2d`).
* **Ingestion Delay (2-day gap):** In a warehouse, recent sales take up to two days to arrive and process.
* **Target Label (60-day forward window):** Checks if the customer makes at least one purchase in the 60 days following the observation date.


**Input:** `mart.fact_sales`, `mart.dim_customer_scd2`.

**Output:** `features.sql` and `mart.training_churn`, built by using six intermediate tables:
- `mart.basket` (provided): `mart.fact_sales` collapsed to one row per purchase, since most features describe visits, not individual products.
- `mart.spine`: the customers and observation dates (the considered population), filtered to customers with recent activity.
- `mart.spine_input`: a copy of `mart.spine`'s keys.  Read by `features.sql`, both for the training cohort and for the scoring cohort in Part 5.
- `mart.feature_events`: The individual past purchases, for each customer and observation date, used to compute features.
- `mart.features`: the seven feature columns, one row per customer and observation date, produced from `mart.feature_events`.
- `mart.target_events` and `mart.label`: the individual future purchases, and the label derived from them.

`mart.training_churn` is then assembled from `mart.spine`, `mart.features`, and `mart.label`, filtered to a complete outcome window, and split into a training set and a test set. `features.sql` is the feature logic itself, written to a file, so it can be run unchanged on a different population later.


### Cell 4.0 — `mart.basket` (Provided)

Each row of `mart.fact_sales` represents one product sold within one purchase, not the purchase as a whole. Most behavioural features describe visits, not individual products: how many purchases a customer made, the average amount spent per purchase, the number of days between two purchases. This cell aggregates `mart.fact_sales` once, to one row per purchase, and excludes lines with no identified customer, for the reason discussed in Q4.



In [24]:
# Cell 4.0 — basket grain
con.execute("""
CREATE OR REPLACE TABLE mart.basket AS
SELECT crm_ref, receipt_id, sale_date, round(sum(amount_net), 2) AS basket_amount
FROM mart.fact_sales
WHERE customer_sk <> -1
GROUP BY crm_ref, receipt_id, sale_date
""")
b = q("""SELECT count(*) AS n_baskets, count(DISTINCT crm_ref) AS n_customers,
                min(sale_date) AS first_day, max(sale_date) AS last_day FROM mart.basket""")
display(b)

,n_baskets,n_customers,first_day,last_day
0,7193,298,2024-01-03,2025-12-31


### Task 4.1 — `mart.spine`: observed customers and dates (Provided)

Build the base table `mart.spine` that lists which customers are eligible to be evaluated on each observation date, along with their profile (segment and city) on that date.  Later tables (`mart.features`, `mart.label`, and `mart.training_churn`) use these rows, ensuring that features and labels are computed on the exact same population.

**Course slides:** Section 5 — *Building the Cohort: Observation Dates and Eligibility*.


**Instructions:** Build `mart.spine` with the columns `crm_ref`, `obs_date`, `segment_at_obs`, and `city_at_obs` using the five observation dates (`2025-04-01`, `2025-07-01`, `2025-08-01`, `2025-10-01`, `2025-11-15`):
1. **Combine dates and customers:** `CROSS JOIN` the five dates with the customers from `mart.basket`.
2. **Apply eligibility:** Keep a customer for a given date only if they made at least one purchase between 365 days and 2 days before `obs_date` `[obs_date - 365 days, obs_date - 2 days[`.
3. **Attach historical customer data:** Use a point-in-time `LEFT JOIN` on `mart.dim_customer_scd2` with `obs_date >= valid_from AND obs_date < valid_to` to retrieve `segment_at_obs` and `city_at_obs`.

**Validation:** The test checks that:
* Each `(crm_ref, obs_date)` pair appears exactly once.
* The table has strictly fewer rows than `total customers × 5 dates` (confirming inactive customers were removed).
* `segment_at_obs` contains no NULL values.

In [25]:
# Task 4.1 — observation cohort with as-of attributes
OBS_DATES = "[DATE '2025-04-01', DATE '2025-07-01', DATE '2025-08-01', DATE '2025-10-01', DATE '2025-11-15']"


con.execute(f"""
CREATE OR REPLACE TABLE mart.spine AS
WITH observation_dates AS (
    SELECT unnest({OBS_DATES}) AS obs_date
),
eligible AS ( -- Combine dates and customers, keep eligible customers
    SELECT DISTINCT b.crm_ref, o.obs_date
    FROM mart.basket b
    CROSS JOIN observation_dates o
    WHERE b.sale_date >= o.obs_date - INTERVAL 365 DAY
      AND b.sale_date <  o.obs_date - INTERVAL 2 DAY
)
SELECT e.crm_ref, e.obs_date, d.segment AS segment_at_obs, d.city AS city_at_obs -- Attach historical customer data
FROM eligible e
LEFT JOIN mart.dim_customer_scd2 d
       ON e.crm_ref   = d.crm_ref
      AND e.obs_date >= d.valid_from
      AND e.obs_date <  d.valid_to
""")


sp = q("""
SELECT count(*) AS n_rows, count(DISTINCT (crm_ref, obs_date)) AS n_keys,
       count(*) FILTER (WHERE segment_at_obs IS NULL) AS n_null_segment,
       (SELECT count(DISTINCT crm_ref) FROM mart.basket) * 5 AS max_possible
FROM mart.spine
""")
display(sp)
display(q("SELECT obs_date, count(*) AS cohort_size FROM mart.spine GROUP BY obs_date ORDER BY obs_date"))

# --- Test
assert sp.n_rows[0] == sp.n_keys[0], "the spine is not at one row per (customer, date)"
assert sp.n_rows[0] < sp.max_possible[0], "the eligibility rule removed nothing"
assert sp.n_null_segment[0] == 0, "an observation date is not covered by any customer version"
print("Test 4.1 passed.")

,n_rows,n_keys,n_null_segment,max_possible
0,1309,1309,0,1490


,obs_date,cohort_size
0,2025-04-01,290
1,2025-07-01,271
2,2025-08-01,262
3,2025-10-01,250
4,2025-11-15,236


Test 4.1 passed.


### Task 4.2 — Create `mart.features` and save it to `features.sql`

Save feature calculation queries in a standalone SQL file (`features.sql`) that reads from `mart.spine_input`, ensuring the exact same code is used for both training and live scoring.

Start by creating the view `mart.spine_input` from `mart.spine`, holding only `crm_ref` and `obs_date`. This view acts as a fixed interface between the cohort and the feature logic, allowing the exact same SQL code in `features.sql` to be used unchanged for both model training and live predictions.

**Course slides:** Section 5 — *Lookback Window Aggregation* and *Sequential Features: LAG*; Section 6 — *Recording Metadata Alongside the Extraction*.

**Goal:** Write two SQL queries into the `features.sql` file:

**4.2a — `SQL_EVENTS`.** Creates `mart.feature_events`, the event-level table, one row per basket usable for one spine row.
`mart.feature_events` is an intermediate table. Saving this intermediate step as a separate table allows for direct testing (e.g Task 4.5 queries `mart.feature_events` directly to confirm that no purchase occurred after the cutoff date).

**4.2b — `SQL_AGGREGATE`.** Creates `mart.features` from `mart.feature_events`, summarizing each customer's list of purchases into fixed numerical values. `mart.feature_events` keeps individual purchases accessible for sorting and testing, while `mart.features` provides the final summarized inputs used by `mart.training_churn` during training and `mart.scoring_churn` during predictions.


**Task 4.2a** — Create `mart.feature_events`

**`SQL_EVENTS`.**
`mart.feature_events` stores one row for each eligible purchase associated to a `(crm_ref, obs_date)` pair. It records the purchase date, the amount, and the number of days since the customer's previous purchase.

It joins two tables:
* `mart.spine_input`: provides the customer and observation date pairs.
* `mart.basket`: provides the purchase history (one row per purchase).


**Instructions.** Join `mart.spine_input` to `mart.basket` on `crm_ref` with the lookback window `[obs_date - 365 days, obs_date - 2 days[`, and add a column `days_since_prev` computed with `lag(sale_date) OVER (PARTITION BY crm_ref, obs_date ORDER BY sale_date)` and `date_diff`. Including `obs_date` in the partition prevents later purchases from being linked to earlier observation periods.


In [29]:
# Task 4.2a — mart.feature_events

con.execute("CREATE OR REPLACE view mart.spine_input AS SELECT crm_ref, obs_date FROM mart.spine")

# ==== YOUR ANSWER START  ====
SQL_EVENTS = """
CREATE OR REPLACE TABLE mart.feature_events AS
SELECT  s.crm_ref,
    s.obs_date, -- debug adding 3
    b.receipt_id,
    b.sale_date,
    b.basket_amount,
       date_diff('day',
                 lag(sale_date) OVER (PARTITION BY s.crm_ref, s.obs_date ORDER BY b.sale_date), --debug
                 b.sale_date) AS days_since_prev
FROM mart.spine_input s
JOIN mart.basket b
  ON  b.crm_ref = s.crm_ref
  AND b.sale_date >= s.obs_date - INTERVAL 365 DAY
  AND b.sale_date <  s.obs_date - INTERVAL 2 DAY;
"""
# ==== YOUR ANSWER END ====


**Task 4.2b** — Create `mart.features`

**Instructions:** Aggregate `mart.feature_events` to one row per `(crm_ref, obs_date)`, with seven summary columns:

* `total_past_orders`: count of purchases in the window.
* `total_past_spend`: sum of purchase amounts.
* `spend_last_30d`: sum of amounts where `sale_date >= obs_date - 30 days`.
* `spend_last_90d`: sum of amounts where `sale_date >= obs_date - 90 days`.
* `avg_basket`: average purchase amount.
* `recency_days`: days between the most recent purchase and `obs_date`.
* `avg_days_between_orders`: average of `days_since_prev`, using `-1.0` when NULL (for customers with only one purchase).

In [30]:
# ==== YOUR ASWER START  ====
SQL_AGGREGATE = """
CREATE OR REPLACE TABLE mart.features AS
SELECT crm_ref, obs_date,
       count(*)  AS total_past_orders,
       ROUND(SUM(basket_amount), 2) AS total_past_spend,
       round(sum(CASE WHEN sale_date >= obs_date - INTERVAL 30 DAY THEN basket_amount ELSE 0 END), 2) AS spend_last_30d,
       round(sum(CASE WHEN sale_date >= obs_date - INTERVAL 90 DAY THEN basket_amount ELSE 0 END), 2) AS spend_last_90d,
       avg(basket_amount)  AS avg_basket,
       date_diff('day', max(sale_date), obs_date) AS recency_days, -- debug
       coalesce(avg(days_since_prev), -1.0) AS avg_days_between_orders
FROM mart.feature_events
GROUP BY crm_ref, obs_date;
"""
# ==== YOUR ASWER END ====

**Task 4.2c** — Write `mart.features` to `features.sql` (Provided)

Write two SQL queries into the `features.sql` file.

**Validation:** check that:
* The output has exactly one row per spine row.
* Every row satisfies the spending condition: `spend_last_30d <= spend_last_90d <= total_past_spend`.


In [31]:
with open("features.sql", "w", encoding="utf-8") as f:
    f.write(SQL_EVENTS + SQL_AGGREGATE)

con.execute(open("features.sql", encoding="utf-8").read())

fc = q("""
SELECT count(*) AS n_rows, count(DISTINCT (crm_ref, obs_date)) AS n_keys,
       (SELECT count(*) FROM mart.spine) AS n_spine,
       count(*) FILTER (WHERE NOT (spend_last_30d <= spend_last_90d
                                   AND spend_last_90d <= total_past_spend)) AS n_inconsistent
FROM mart.features
""")
display(fc)
display(q("SELECT * FROM mart.features ORDER BY crm_ref, obs_date LIMIT 5"))

# --- Test
assert fc.n_rows[0] == fc.n_keys[0], "the feature table is not at one row per (customer, date)"
assert fc.n_rows[0] == fc.n_spine[0], "some spine rows have no features"
assert fc.n_inconsistent[0] == 0, "nested windows are inconsistent, check the sale date conditions"
print("Test 4.2 passed. features.sql written and executed.")

,n_rows,n_keys,n_spine,n_inconsistent
0,1309,1309,1309,0


,crm_ref,obs_date,total_past_orders,total_past_spend,spend_last_30d,spend_last_90d,avg_basket,recency_days,avg_days_between_orders
0,CLI-4001,2025-04-01,20,200.05,0.0,3.4,10.002500,39,16.157895
1,CLI-4001,2025-07-01,20,202.25,26.5,83.2,10.112500,13,18.000000
2,CLI-4001,2025-08-01,20,207.35,16.5,70.3,10.367500,4,18.421053
3,CLI-4001,2025-10-01,17,148.70,4.3,51.6,8.747059,4,20.187500
4,CLI-4001,2025-11-15,21,179.30,30.6,55.4,8.538095,7,16.700000


Test 4.2 passed. features.sql written and executed.


### Task 4.3 — Create `mart.label`

Record whether each customer in `mart.spine` made a purchase in the 60 days following their observation date.

**Course slides:** Section 5 — *Target Label Construction*.

If the past and future time windows overlap, future data will leak into the training features.

**Instructions:** Run two queries:

1. **Find future purchases (`mart.target_events`):**
   stores one row for each purchase made during the 60-day window following an observation date. It records the `receipt_id` and `sale_date`. Saving this intermediate table allows Task 4.5 to test the data directly and confirm that no purchase occurred before `obs_date`.
   * Join `mart.spine` to `mart.basket` on `crm_ref` where `sale_date >= obs_date` and `sale_date < obs_date + INTERVAL '60' DAY`.
   * Keep `crm_ref`, `obs_date`, `receipt_id`, and `sale_date`.

2. **Calculate the label (`mart.label`):**
   * `LEFT JOIN` `mart.spine` to `mart.target_events` on `(crm_ref, obs_date)`.
   * Set `target_repurchase_60d` to `1` if the customer made at least one purchase, and `0` otherwise. The `LEFT JOIN` ensures non-buyers are assigned `0` rather than excluded.

**Validation:**
* The proportion of customers with `target_repurchase_60d = 1` is strictly between 0.2 and 0.8.

In [37]:
# Task 4.3 — forward window and label
# ==== YOUR ANSWER START  ====
con.execute(""" -- Find future purchases
CREATE OR REPLACE TABLE mart.target_events AS -- can contain multiple purchases in the future
select s.crm_ref, obs_date, receipt_id, sale_date
FROM mart.spine s
JOIN mart.basket b
  ON  s.crm_ref = b.crm_ref
  AND b.sale_date >= s.obs_date
  AND b.sale_date <  s.obs_date + INTERVAL 60 DAY;
""")

con.execute(""" -- Calculate the label
CREATE OR REPLACE TABLE mart.label AS
SELECT s.crm_ref, s.obs_date,
       CASE WHEN count(t.crm_ref) > 0 then 1 else 0 end AS target_repurchase_60d -- debug
FROM mart.spine s
LEFT JOIN  mart.target_events t on s.crm_ref = t.crm_ref and s.obs_date = t.obs_date
GROUP BY s.crm_ref, s.obs_date; -- possible multible purchases -> 0/1 label for one row
""")
# ==== YOUR ANSWER END ====

lb = q("""SELECT obs_date, count(*) AS n_rows, round(avg(target_repurchase_60d), 3) AS positive_rate
          FROM mart.label GROUP BY obs_date ORDER BY obs_date""")
display(lb)
overall = con.execute("SELECT avg(target_repurchase_60d) FROM mart.label").fetchone()[0]
print("overall positive rate:", round(overall, 3))

# --- Test
assert con.execute("SELECT count(*) FROM mart.label").fetchone()[0] == con.execute("SELECT count(*) FROM mart.spine").fetchone()[0]
assert 0.2 < overall < 0.8, "degenerate target, check the join operation"
print("Test 4.3 passed.")

,obs_date,n_rows,positive_rate
0,2025-04-01,290,0.634
1,2025-07-01,271,0.624
2,2025-08-01,262,0.615
3,2025-10-01,250,0.572
4,2025-11-15,236,0.576


overall positive rate: 0.606
Test 4.3 passed.


### Cell 4.1 — `mart.training_churn` (Provided)

Build the final table `mart.training_churn` by joining customer segments, features, and labels into a single permanent dataset, then assigning rows to a train or test set.

It is assembled from three tables built previously. `mart.spine` supplies the population of `(crm_ref, obs_date)` pairs and the customer's segment as of that date. `mart.features` supplies the seven point-in-time feature columns, computed strictly from purchases before `obs_date - 2 days`. `mart.label` supplies the outcome, whether the customer bought again in the 60 days following `obs_date`.


**Course slides:** Section 5 — *Assembling the Labelled Table* and *Temporal Train-Test Split*; Section 0 — *Warehouse storage of the training set*.

**Instructions:** Create `mart.training_churn` with columns `crm_ref`, `obs_date`, `segment_at_obs`, the seven feature columns, `y_repurchase_60d`, and `split_group`:

1. **Join tables:** Join `mart.spine`, `mart.features`, and `mart.label` on `(crm_ref, obs_date)`.
2. **Remove incomplete label windows:** The label needs 60 days of history after `obs_date`. Find the latest date dynamically (`SELECT MAX(sale_date) FROM mart.fact_sales`). Keep only rows where `obs_date + INTERVAL '60' DAY <= MAX(sale_date)`. This removes `2025-11-15`. This drops `2025-11-15`.
3. **Split into train and test sets:** With `TRAIN_END = DATE '2025-07-01'`:
   * Assign `split_group = 'TRAIN'` for rows where `obs_date <= TRAIN_END`.
   * Assign `split_group = 'TEST'` for rows where `obs_date >= TRAIN_END + INTERVAL '60' DAY`.
   * Drop rows falling inside the 60-day buffer period. This drops `2025-08-01`.


**Validation:** The test checks that:
* `2025-11-15` is not present in the table.
* `2025-08-01` is not present in the table.
* Both `TRAIN` and `TEST` are non-empty.

Two of the five observation dates were removed. Of the three remaining dates, two, `2025-04-01` and `2025-07-01`, satisfy `obs_date <= TRAIN_END` and are both classified TRAIN, each contributing its own set of rows. Observations at  `2025-10-01` are in TEST.

In [38]:
# Cell 4.1 — build mart.training_churn

con.execute("""
CREATE OR REPLACE TABLE mart.training_churn AS
WITH horizon AS (SELECT max(sale_date) AS h FROM mart.fact_sales),
assembled AS (
    SELECT s.crm_ref, s.obs_date, s.segment_at_obs,
           f.total_past_orders, f.total_past_spend, f.spend_last_30d, f.spend_last_90d,
           f.avg_basket, f.recency_days, f.avg_days_between_orders,
           l.target_repurchase_60d AS y_repurchase_60d,
           CASE WHEN s.obs_date <= DATE '2025-07-01'                     THEN 'TRAIN'
                WHEN s.obs_date >= DATE '2025-07-01' + INTERVAL 60 DAY   THEN 'TEST'
                ELSE 'BUFFER' END AS split_group
    FROM mart.spine s
    JOIN mart.features f USING (crm_ref, obs_date)
    JOIN mart.label    l USING (crm_ref, obs_date)
    CROSS JOIN horizon
    WHERE s.obs_date + INTERVAL 60 DAY <= horizon.h
)
SELECT * FROM assembled WHERE split_group <> 'BUFFER'
""")

tc = q("""SELECT split_group, obs_date, count(*) AS n_rows, round(avg(y_repurchase_60d), 3) AS positive_rate
          FROM mart.training_churn GROUP BY 1, 2 ORDER BY 2""")
display(tc)
kept = set(tc.obs_date.astype(str))

# --- Test
assert '2025-11-15' not in kept, "an observation date with an incomplete outcome window was kept"
assert '2025-08-01' not in kept, "an observation date inside the buffer was kept"
assert set(tc.split_group) == {'TRAIN', 'TEST'}, "both partitions must be present"
assert con.execute("SELECT count(*) FROM mart.training_churn WHERE y_repurchase_60d IS NULL").fetchone()[0] == 0, \
    "a row has no label, the join to mart.label is dropping or missing rows for some (crm_ref, obs_date) pairs"
print("Test 4.4 passed. mart.training_churn is a persistent table in the warehouse.")

,split_group,obs_date,n_rows,positive_rate
0,TRAIN,2025-04-01,290,0.634
1,TRAIN,2025-07-01,271,0.624
2,TEST,2025-10-01,250,0.572


Test 4.4 passed. mart.training_churn is a persistent table in the warehouse.


**Q6.** Sales history stops on 2025-12-31. For an observation dated 2025-11-15, would `target_repurchase_60d` be computed correctly if this date were not excluded? Explain why, and why the resulting error is worse than a missing value. Then explain why the buffer between the TRAIN and TEST sets must be at least as long as the 60-day label window, and how the measured test performance would be affected without it.

> **YOUR ANSWER.**
>

### Cell 4.2 — Test of Temporal Boundaries

 Verify that feature and label tables respect their date boundaries by running validation checks on both pipeline outputs and a table `mart.feature_events_broken`, an event table with `sale_date <= obs_date` instead of `sale_date < obs_date - 2`.

**Course slides:** Section 5 — *Testing Temporal Boundaries*.

**Instructions:**

1. **Write the feature boundary check:** Count the number of rows where `sale_date >= obs_date - INTERVAL '2' DAY`, in `mart.feature_events` and in the deliberately flawed table `mart.feature_events_broken`.
2. **Write the label boundary check:** Count the number of rows in `mart.target_events` where `sale_date < obs_date`.
3. **Run the checks:** all three counts should be computed and printed before the final test.


**Validation:** The test passes when:
* `mart.feature_events_broken` reports violations (count > 0).
* `mart.feature_events` reports zero violations (count = 0).
* `mart.target_events` reports zero violations (count = 0).


In [ ]:
# Cell 4.2 — leakage test
con.execute("""
CREATE OR REPLACE view mart.feature_events_broken AS
SELECT s.crm_ref, s.obs_date, b.receipt_id, b.sale_date
FROM mart.spine s
JOIN mart.basket b
  ON  b.crm_ref   = s.crm_ref
  AND b.sale_date >= s.obs_date - INTERVAL 365 DAY
  AND b.sale_date <= s.obs_date
""")

# ==== YOUR ANSWER START  ====
v_broken = con.execute("""
    SELECT count(*) FROM mart.feature_events_broken
    WHERE ...
""").fetchone()[0]

v_features = con.execute("""
    SELECT count(*) FROM mart.feature_events
    WHERE ...
""").fetchone()[0]

n_past_events_in_label = con.execute("""
    SELECT count(*) FROM mart.target_events
    WHERE ...
""").fetchone()[0]
# ==== YOUR ANSWER END  ====

print("violations, broken table:", v_broken)
print("violations, feature events:", v_features)
print("label events before obs_date:", n_past_events_in_label)

# --- Test
assert v_broken > 0, "the test cannot detect a known error, it is not testing anything"
assert v_features == 0, "feature events cross the cutoff"
assert n_past_events_in_label == 0, "label events cross the cutoff backwards"
print("Test 4.5 passed.")

# Part 5 — Scoring, Run Tracking, Temporal Leakage

**Course slides:** Section 5 — *Constructing the Scoring Table*; Section 6 — *Recording Metadata Alongside the Extraction*, *Using the Extraction Manifest*, and *Synthesis: Production Parity and Provenance*.

**Input:** `features.sql`, `mart.basket`, `mart.dim_customer_scd2`, `mart.training_churn`.

**Output:** `mart.scoring_churn` and `extraction.manifest.json`, built through two intermediate tables:
- `mart.spine_input`: overwritten with a single scoring date, built directly from `mart.basket` rather than from `mart.spine`, since there is no training cohort to copy from at this stage.
- `mart.feature_events` and `mart.features`: recomputed by the same `features.sql`, unchanged, this time from the scoring row in `mart.spine_input`.

`mart.scoring_churn` is then assembled from `mart.spine_input`, `mart.features`, and `mart.dim_customer_scd2` (for the customer's segment as of the scoring date), with no label and no split assignment. `extraction.manifest.json` records the fingerprint of `features.sql` and the state of `mart.fact_sales` at the time this table was built.


This part also compares two models trained on `mart.training_churn`: a clean model and  one trained on a feature that leaks the future. Comparing their AUC scores measures the exact performance loss once the leaked future signal is no longer available at inference.

### Task 5.1 — `mart.scoring_churn`

Build `mart.scoring_churn` for scoring date `2026-01-15` by executing the unmodified `features.sql` file on a live cohort with no labels. Running the exact same feature file guarantees that the input structure fed to the model remains identical.

**Course slides:** Section 5 — *Constructing the Scoring Table*, Section 6 — *Production Parity*.

**Instructions:** Use the scoring date `2026-01-15`:

1. **Define the view `mart.spine_input`:** Set `obs_date` to `2026-01-15`. Insert distinct customers from `mart.basket` who made at least one purchase in the window `[obs_date - INTERVAL '365' DAY, obs_date - INTERVAL '2' DAY[` (same eligibility window used for the training cohort).
2. **Execute `features.sql`:** Run the file directly from disk without altering any of its queries.
3. **Assemble `mart.scoring_churn`:** build the final scoring table using the same logic as `mart.training_churn`, minus the two columns exclusive to training. Source tables/views:
* `mart.spine_input`: provides customer identifiers and the scoring date.
* `mart.features`: provides the seven feature columns newly computed by `features.sql`.
* `mart.dim_customer_scd2`: provides each customer's active segment on the scoring date via an as-of join.

Excluded columns: **`y_repurchase_60d`** label and the split column **(`split_group`):**.

**Validation:** :
* The column set matches `mart.training_churn` exactly, minus `y_repurchase_60d` and `split_group`.
* Feature columns contain computed figures, not default or empty values.

In [ ]:
# Task 5.1 — scoring table, produced by the same features.sql
SCORE_DATE = "DATE '2026-01-15'"

# ==== YOUR ANSWER START  ====
con.execute(f""" -- Populate mart.spine_input
CREATE OR REPLACE view mart.spine_input AS
SELECT DISTINCT b.crm_ref, {SCORE_DATE} AS obs_date
FROM ...
WHERE ...
""")

con.execute(open("features.sql", encoding="utf-8").read())

con.execute(""" -- Assemble mart.scoring_churn
CREATE OR REPLACE TABLE mart.scoring_churn AS
SELECT ...
       f.* EXCLUDE (crm_ref, obs_date) -- crm_ref, obs_date already present from mart.spine_input
FROM ...
JOIN ...
LEFT JOIN ...
""")
# ==== YOUR ANSWER END ====
sc = q("SELECT count(*) AS n_rows, sum(total_past_orders) AS total_orders FROM mart.scoring_churn")
display(sc)
display(q("SELECT * FROM mart.scoring_churn LIMIT 3"))

train_cols = set(q("SELECT * FROM mart.training_churn LIMIT 0").columns) - {"y_repurchase_60d", "split_group"}
score_cols = set(q("SELECT * FROM mart.scoring_churn LIMIT 0").columns)

# --- Test
assert sc.n_rows[0] > 0, "empty scoring cohort"
assert sc.total_orders[0] > 0, "every feature is at its default value, the join key is wrong"
assert train_cols == score_cols, f"column mismatch: {sorted(train_cols ^ score_cols)}"
print("Test 5.1 passed. Training and scoring share the same columns and the same SQL file.")

### Cell 5.0 — `extraction.manifest.json` (Provided)

Generate an extraction manifest documenting the exact code version, execution timestamp, dataset summary metrics, and source table state.

**Course slides:** Section 6 — *Recording Metadata Alongside the Extraction*,  *Using the Extraction Manifest*.

**Problem:** Warehouse data changes over time as late orders arrive, bad data gets fixed, and tables reload. Running the exact same SQL script later can give different numbers. Without a saved run record, you cannot tell if a change in model performance comes from updated code or modified source data.

**Instructions:** Create `extraction.manifest.json` containing the following fields:

| Field | Description | Format / Source |
|---|---|---|
| `executed_at` | Run execution timestamp | ISO 8601 string |
| `query_sha` | Hash of `features.sql`'s raw bytes | SHA-256 hex digest (64 characters) computed from file bytes |
| `n_rows` | Total row count | `COUNT(*)` from `mart.training_churn` |
| `sum_total_past_spend` | Aggregate past spend | `SUM(total_past_spend)` from `mart.training_churn`, rounded to 2 decimal places |
| `source_max_sale_date` | Latest transaction date | `MAX(sale_date)` from `mart.fact_sales`, formatted as a string |
| `source_fact_rows` | Total source transaction count | `COUNT(*)` from `mart.fact_sales` |


The first two fields record the code version, the following four record the state of the data.


**Validation:**
* `extraction.manifest.json` exists on disk.
* `query_sha` is a valid 64-character hexadecimal string.
* Output metrics match values queried directly from `mart.training_churn` and `mart.fact_sales`.





In [ ]:
# Cell 5.0 — extraction manifest
stats = con.execute("""
SELECT count(*) AS n_rows, round(sum(total_past_spend), 2) AS sum_spend FROM mart.training_churn
""").fetchone()
src = con.execute("SELECT max(sale_date), count(*) FROM mart.fact_sales").fetchone()

manifest = {
    "executed_at":          dt.datetime.now().isoformat(timespec="seconds"),
    "query_sha":            hashlib.sha256(open("features.sql", "rb").read()).hexdigest(),
    "n_rows":               int(stats[0]),
    "sum_total_past_spend": float(stats[1]),
    "source_max_sale_date": str(src[0]),
    "source_fact_rows":     int(src[1]),
}
with open("extraction.manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

# --- Test
assert os.path.exists("extraction.manifest.json")
assert len(manifest["query_sha"]) == 64
print("Test 5.2 passed.")


**Q7.** The same extraction is run again later. A final output changes unexpectedly (such as a report total, a model evaluation score, or a dashboard figure) with no obvious cause. Using only the six fields of the manifest, from this run and the previous one, describe what each of the three following situations tells about the cause, and what to check next in each case.
(a) same `query_sha`, different `n_rows` and `source_fact_rows`.
(b) different `query_sha`, same `n_rows` and same `source_fact_rows`.
(c) everything identical.

> **YOUR ANSWER.**
> (a) ...

> (b) ...
> (c) ...

**Q8.** The pipeline generates two final tables: `mart.training_churn` and `mart.scoring_churn`. List the validation test in this lab and the specific issue each test prevents. Then, identify one critical data property that remains unverified by every gate.

> **YOUR ANSWER:**
>
> **Test 1.1:** ... **Test 1.2:** ... **Test 1.3:** ... **Test 2.2d:** ... **Test 2.3:** ... **Test 3.1a:** ...
**Test 3.1b:** ... **Test 3.2:** ... **Test 4.1:** ... **Test 4.2:** ... **Test 4.3:** ...
**Test 4.4:** ...
**Test 4.5:** ... **Test 5.1:** ...
>
> **Unverified property:**
> ...


### Cell 5.1 — Demonstration of Temporal Leakage

Compare three evaluations on the exact same test records, using a feature built with the wrong date bounds.

**Course slides**  Section 4 - *Temporal Leakage: Using Future State at Observation Date*, Section 6: *Production Parity*


`spend_next_60d` is an ordinary sum of customer purchases from `mart.basket`, computed over `[obs_date, obs_date + 60 days[`, the same window used to build the label in Task 4.3.

Evaluate three setups on the exact same TEST rows, using the AUC score for each:

| Setup | Trained on | Evaluated on | What it shows |
| :--- | :--- | :--- | :--- |
| **Model A** | Clean, point-in-time features | Same clean features | The model's real performance |
| **Model B, leaked feature with its real value** | Clean features plus `spend_next_60d` | Same features, `spend_next_60d` at its real value | The score reported if the leak was unnoticed |
| **Model B, leaked feature unavailable** | Same Model B, not retrained | Same features, `spend_next_60d` set to 0 | The score reported once the leaked signal is removed |


**Validation**
* The AUC of "Model B, leaked feature with its real value" is strictly greater than the AUC of Model A.
* The AUC of "Model B, leaked feature unavailable" is strictly lower than the AUC of Model A.






In [ ]:
# Cell 5.1 — clean model, leaked model offline, leaked model, evaluated at scoring time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

np.random.seed(0)

# Reload mart.training_churn and inject the leaked feature spend_next_60d.
# This column computes total customer spend over the 60 days following obs_date
# calculated from mart.target_events and mart.basket.
# using mart.target_events and mart.basket, using the outcome window of y_repurchase_60d.
df = con.execute("""
SELECT t.*, coalesce(l.future_spend, 0.0) AS spend_next_60d -- sets spend to zero for customers without future activity
FROM mart.training_churn t
LEFT JOIN (
    SELECT e.crm_ref, e.obs_date, sum(b.basket_amount) AS future_spend
    FROM mart.target_events e
    JOIN mart.basket b USING (crm_ref, receipt_id, sale_date)
    GROUP BY e.crm_ref, e.obs_date
) l USING (crm_ref, obs_date)
""").df()
df["is_vip"] = (df.segment_at_obs == "VIP").astype(int)

CLEAN = ["total_past_orders", "total_past_spend", "spend_last_30d", "spend_last_90d",
         "avg_basket", "recency_days", "avg_days_between_orders", "is_vip"]
LEAKED = CLEAN + ["spend_next_60d"]

tr = df[df.split_group == "TRAIN"]
te = df[df.split_group == "TEST"]
print(f"train rows {len(tr)}, test rows {len(te)}")

# auc_a: Baseline model performance, trained and evaluated on the eight validated,
# historical features without any event occurring after obs_date.
model_a = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0).fit(tr[CLEAN], tr.y_repurchase_60d)
p_a = model_a.predict_proba(te[CLEAN])[:, 1]
auc_a = roc_auc_score(te.y_repurchase_60d, p_a)

# auc_b_offline: score resulting from training and testing
# the model with spend_next_60d included.
model_b = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0).fit(tr[LEAKED], tr.y_repurchase_60d)
p_b_leaked = model_b.predict_proba(te[LEAKED])[:, 1]
auc_b_leaked = roc_auc_score(te.y_repurchase_60d, p_b_leaked)

# auc_b_scoring: scoring performance of model_b with spend_next_60d set to 0
te_scoring = te.copy()
te_scoring["spend_next_60d"] = 0.0
p_b_scoring = model_b.predict_proba(te_scoring[LEAKED])[:, 1]
auc_b_scoring = roc_auc_score(te.y_repurchase_60d, p_b_scoring)

print(f"A, clean features only       AUC = {auc_a:.3f}")
print(f"B, leaked feature at its real value     AUC = {auc_b_leaked:.3f}")
print(f"B, leaked feature unavailable   AUC = {auc_b_scoring:.3f}")

plt.figure(figsize=(7.5, 5.5))
for p, lab, style in [(p_a, f"A clean (AUC {auc_a:.3f})", "-"),
                      (p_b_leaked, f"B, leaked feature at its real value (AUC {auc_b_leaked:.3f})", "--"),
                      (p_b_scoring, f"B, leaked feature unavailable (AUC {auc_b_scoring:.3f})", ":")]:
    fpr, tpr, _ = roc_curve(te.y_repurchase_60d, p)
    plt.plot(fpr, tpr, style, lw=2.2, label=lab)
plt.plot([0, 1], [0, 1], color="grey", lw=1)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("Same test partition, three evaluations")
plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

assert auc_b_leaked > auc_a, "the leaked model should score higher than the clean model"
assert auc_b_scoring < auc_a, "the leaked model should be worse than the clean one"
print("Test 5.3 passed.")

**Q9.** Evaluated on identical test records, Model B outperforms Model A when using actual `spend_next_60d` values, but underperforms Model A when `spend_next_60d` is set to 0.

1. Explain this behavior in terms of observable information at `obs_date`.
2. For both `spend_next_60d` and `recency_days`, answer the following: What does it measure? Over which period is it computed? Is it known at inference time?

> **YOUR ANSWER.**
>...
